# Notebook 03 v25 — Consolidated Historical + Baseline Feature Engineering & Pretraining Handoff

This consolidated master notebook replaces the separate `v23` and `v24` feature engineering notebooks:
1. **Preserves the Frozen Cohorts (`2016–2019` + `2020+`):** Loads the expanded `N2_DIR` historical run and the completed `BASELINE_N3_DIR` (`v23`) run, verifying all SHA-256 input checksums and confirming exact equality on all 64 inherited predictors before adding new features.
2. **Engineers Causal v25 Alpha & Disqualification Features Across the Unified Panel:**
   - **Path-Dependent "Scar Tissue" Metrics:** 6-month and 12-month **Ulcer Index** (`price_ulcer_index_6m`, `price_ulcer_index_12m`), **Chronic Underwater Duration** (`price_underwater_duration_20pct_12m`, `price_underwater_duration_40pct_12m`), **Acute-vs-Chronic Drawdown Ratio**, **3-Month Drawdown Velocity**, and **Excess Drawdown vs. VOO**.
   - **Bounded Volatility Asymmetry & Term Structure:** Replaces collinear raw volatility levels with **Downside Volatility Share** (`price_downside_vol_ratio_252d`), **Idiosyncratic Volatility Share** (`price_idio_vol_ratio_252d`), **Volatility Term Structure** (`63d/252d` and `126d/252d`), and **Cross-Sectional Volatility Relative to VOO**.
   - **Multi-Horizon Momentum & 2nd-Derivative Price Curvature:** Adds `12m-ex-3m` total and VOO-relative momentum, `21d` and `63d` **Return Trajectory Curvature** relative to the 252-day geometric pace, **QoQ Price Return Acceleration**, and the **Martin (Ulcer-Adjusted) Efficiency Ratio**.
   - **2nd-Derivative Fundamental Velocity (`3m` QoQ & `6m` Semi-Annual):** Computes causal 3-month and 6-month acceleration across YoY growth metrics and 3-month trajectory deltas across TTM margins, ROA, ROE, cash conversion, accruals, leverage, and interest coverage.
   - **Cross-Domain "Falling Knife" vs. "Coiled Spring" Interactions:** Interacts drawdown magnitude with YoY cash-to-assets burn, QoQ operating cash flow acceleration, operating margin trajectory, leverage stress, and short-term (`21d`/`63d`) price stabilization.
3. **Quarantines Regime-Overfitting Predictors:** Removes the 6 constant-per-date `market_voo_*` macro date-stamp columns and the 4 raw volatility level columns from the active `feature_columns` whitelist while preserving `baseline_v24_feature_columns` in `pretraining_contract.json` for apples-to-apples ablation in Notebook 04.

## 1. Locate the completed upstream runs

`N2_DIR` points to the expanded N2 historical run (`2016–2019`). `BASELINE_N3_DIR` points to the completed October 15 `v23` baseline run (`2020+`). `PRIOR_V24_DIR` (if present) is used to cross-verify the exact `input_sha256` manifest from `v24`.

In [1]:
from pathlib import Path
import hashlib
import json
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name.lower() == 'notebooks' else HERE

N2_DIR = PROJECT_ROOT / 'data/interim/audits/nb02_history_v3/20261020T145055491621Z'
BASELINE_N3_DIR = PROJECT_ROOT / 'data/interim/audits/nb03_revision/2026-10-15-v23_20261015T225833974213Z'
PRIOR_V24_DIR = PROJECT_ROOT / 'data/interim/audits/nb03_history_v24/20261020T152222962533Z'
VOO_PATH = PROJECT_ROOT / 'data/raw/market/VOO_dividend_adjusted_2010_plus.parquet'

VERSION = 'nb03_v25_SCAR_TISSUE_VELOCITY'

# Quarantine switches for the active FEATURE_COLUMNS whitelist exported to Notebook 04:
QUARANTINE_MACRO_VOO_DATE_STAMPS = True
QUARANTINE_RAW_VOLATILITY_LEVELS = True

print('Project root:', PROJECT_ROOT)
print('Expanded N2:', N2_DIR, '| exists:', N2_DIR.is_dir())
print('Completed v23 baseline:', BASELINE_N3_DIR, '| exists:', BASELINE_N3_DIR.is_dir())
print('Prior v24 reference:', PRIOR_V24_DIR, '| exists:', PRIOR_V24_DIR.is_dir())

Project root: /Users/neilyejjey/stock_signal_engine_v2
Expanded N2: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb02_history_v3/20261020T145055491621Z | exists: True
Completed v23 baseline: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb03_revision/2026-10-15-v23_20261015T225833974213Z | exists: True
Prior v24 reference: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb03_history_v24/20261020T152222962533Z | exists: True


## 2. Historical construction and validation helpers

Each selected statement and every trailing/prior-year quarter must have been available at the observation date. TTM uses four consecutive fiscal quarters and rejects conflicted dependencies and issuer/currency mismatches. Late updates to an older quarter cannot replace the newest period selected by N2.

Price calculations use the historical cached dividend-adjusted series throughout each window, never a splice to the old baseline snapshot. All observed VOO sessions are retained in the grid; missing stock sessions remain missing. Exact historical labels require a complete observed price path through the month-end twelve months later.

In [2]:
KEYS = ['security_key', 'as_of_date']
TARGET = 'target_excess_return_12m'

def sha_file(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json(x, p):
    Path(p).write_text(json.dumps(x, indent=2, default=str))

def require(condition, message):
    if not condition:
        raise ValueError(message)

def numeric(r, f):
    try:
        v = float(r.get(f, np.nan))
        return v if np.isfinite(v) else np.nan
    except (ValueError, TypeError):
        return np.nan

def fiscal_ordinal(r):
    try:
        q = str(r['period'])
        y = float(r['fiscalYear'])
        return int(y) * 4 + int(q[1]) - 1 if q in ['Q1', 'Q2', 'Q3', 'Q4'] and y.is_integer() else None
    except (ValueError, TypeError, KeyError):
        return None

def build_financial_inputs(panel, ledger):
    """Resolve every dependency at the observation date, including revised prior quarters."""
    ledger = ledger.copy()
    for c in ['date', 'tradable_from']:
        ledger[c] = pd.to_datetime(ledger[c], errors='raise')
    require(not ledger.ledger_row_id.duplicated().any(), 'Duplicate historical ledger IDs')
    rows = ledger.to_dict('records')
    lookup = {int(r['ledger_row_id']): r for r in rows}
    groups = {}
    for r in rows:
        if pd.notna(r['tradable_from']):
            groups.setdefault((r['security_key'], r['statement_type']), []).append(r)
    for g in groups.values():
        g.sort(key=lambda r: (r['tradable_from'], r['date'], r['ledger_row_id']))
    output = []
    lineage = []
    for key, left in panel.groupby('security_key', sort=False):
        state = {k: {} for k in STATEMENT_FIELD_REGISTRY}
        pointers = {k: 0 for k in state}
        for observation in left.sort_values('as_of_date').to_dict('records'):
            d = observation['as_of_date']
            out = dict(observation)
            for kind, fields in STATEMENT_FIELD_REGISTRY.items():
                events = groups.get((key, kind), [])
                pos = pointers[kind]
                while pos < len(events) and events[pos]['tradable_from'] <= d:
                    event = events[pos]
                    ordinal = fiscal_ordinal(event)
                    if ordinal is not None:
                        state[kind][ordinal] = event
                    pos += 1
                pointers[kind] = pos
                selected_id = observation.get(kind + '__ledger_row_id')
                selected = lookup.get(int(selected_id)) if pd.notna(selected_id) else None
                if selected is not None:
                    require(selected['security_key'] == key and selected['statement_type'] == kind, 'Wrong selected issuer/type')
                    require(selected['tradable_from'] <= d and selected['dependency_usable'], 'Invalid selected dependency')
                end = fiscal_ordinal(selected) if selected else None

                def get(ordinal):
                    r = state[kind].get(ordinal)
                    if r is None or not r['dependency_usable']:
                        return None
                    if selected and (
                        r.get('normalized_cik') != selected.get('normalized_cik')
                        or r.get('reportedCurrency') != selected.get('reportedCurrency')
                    ):
                        return None
                    return r

                def record_dependency(r, role):
                    if r is not None:
                        lineage.append(
                            dict(
                                security_key=key,
                                as_of_date=d,
                                statement_type=kind,
                                dependency_role=role,
                                ledger_row_id=int(r['ledger_row_id']),
                                tradable_from=r['tradable_from'],
                            )
                        )

                for f in fields:
                    out[kind + '__' + f] = numeric(selected or {}, f)
                record_dependency(selected, 'selected')
                prior = get(end - 4) if end is not None else None
                for f in fields:
                    out[kind + '__' + f + '_prior_year'] = numeric(prior or {}, f)
                record_dependency(prior, 'prior_year_quarter')
                if kind == 'income':
                    for f in ['weightedAverageShsOut', 'weightedAverageShsOutDil']:
                        out['income__' + f + '_prior_year_quarter'] = numeric(prior or {}, f)
                if kind == 'balance':
                    continue
                for offset, suffix in [(0, ''), (4, '_prior_year')]:
                    components = [get(end - offset - j) for j in range(4)] if end is not None else [None] * 4
                    valid = all(r is not None for r in components)
                    if valid:
                        dates = [r['date'] for r in components]
                        gaps = [(dates[j] - dates[j + 1]).days for j in range(3)]
                        valid = all(45 <= gap <= 135 for gap in gaps) and 240 <= (dates[0] - dates[-1]).days <= 330
                    out[kind + '__ttm_valid' + suffix] = valid
                    for j, r in enumerate(components):
                        if valid:
                            record_dependency(r, 'ttm' + suffix + '_q' + str(j))
                    for f in (INCOME_TTM_FIELDS if kind == 'income' else CASHFLOW_TTM_FIELDS):
                        values = [numeric(r or {}, f) for r in components]
                        out[kind + '__' + f + '_ttm' + suffix] = sum(values) if valid and np.isfinite(values).all() else np.nan
            output.append(out)
    out = pd.DataFrame(output).sort_values(KEYS).reset_index(drop=True)
    dep = pd.DataFrame(lineage, columns=KEYS + ['statement_type', 'dependency_role', 'ledger_row_id', 'tradable_from'])
    require((dep.tradable_from <= dep.as_of_date).all(), 'Future financial dependency')
    return out, dep

def historical_market_and_labels(panel, prices, voo, overlap, feature_columns):
    """No snapshot stitching and no forward filling. Labels never affect feature construction."""
    prices = prices.loc[prices.price_usable.astype(bool)].copy()
    prices['date'] = pd.to_datetime(prices.date)
    voo = voo.copy()
    voo['date'] = pd.to_datetime(voo.date)
    require(not prices.duplicated(['security_key', 'date']).any(), 'Duplicate usable historical prices')
    require(not voo.date.duplicated().any(), 'Duplicate VOO sessions')
    market = voo.set_index('date').adjClose.sort_index().astype(float)
    require((np.isfinite(market) & market.gt(0)).all(), 'Invalid VOO adjusted prices')
    sessions = market.index
    month_ends = pd.Series(sessions, index=sessions.to_period('M')).groupby(level=0).max()
    price_groups = {k: g.set_index('date').adjClose.astype(float) for k, g in prices.groupby('security_key')}
    failed = set(overlap.loc[~overlap.constant_scale_diagnostic_pass.astype(bool), 'security_key'])
    compared = set(overlap.security_key)
    outputs = []
    audits = []
    for count, (key, left) in enumerate(panel.groupby('security_key', sort=False), 1):
        series = price_groups.get(key, pd.Series(dtype=float)).reindex(sessions)
        grid = pd.DataFrame({'security_key': key, 'date': sessions, 'stock_adj_close': series.to_numpy(), 'voo_adj_close': market.to_numpy()})
        daily = build_security_price_features(grid).set_index('date')
        complete = series.notna().rolling(253, min_periods=253).sum().eq(253)
        for r in left.to_dict('records'):
            d = r['as_of_date']
            features = daily.loc[d]
            out = {**{k: r[k] for k in KEYS}, **{f: features[f] for f in feature_columns}}
            out['price_history_complete'] = bool(complete.loc[d])
            out['price_snapshot_status'] = (
                'overlap_conflict_hold'
                if key in failed
                else 'overlap_scale_consistent'
                if key in compared
                else 'independent_historical_series_no_overlap'
            )
            target = month_ends.get(d.to_period('M') + 12, pd.NaT)
            out['target_end_date_12m'] = target
            out[TARGET] = np.nan
            out['observation_kind'] = 'unknown'
            out['supervision_available_date'] = pd.NaT
            reason = 'missing_price_endpoint_or_path'
            if pd.isna(target):
                out['observation_kind'] = 'immature'
                reason = 'target_beyond_calendar'
            else:
                path = series.loc[d:target]
                complete_path = len(path) > 0 and path.notna().all() and path.gt(0).all()
                if key in failed:
                    reason = 'price_snapshot_conflict'
                elif complete_path:
                    stock_return = path.iloc[-1] / path.iloc[0] - 1
                    benchmark_return = market.loc[target] / market.loc[d] - 1
                    out[TARGET] = stock_return - benchmark_return
                    out['observation_kind'] = 'exact'
                    out['supervision_available_date'] = target
                    reason = 'observed_adjusted_prices_continuous_path'
            out['historical_outcome_reason'] = reason
            outputs.append(out)
        audits.append(
            dict(
                security_key=key,
                price_snapshot_status=out['price_snapshot_status'],
                available_sessions=int(series.notna().sum()),
                total_months=len(left),
            )
        )
        if count % 100 == 0:
            print('Historical price histories processed:', count)
    return pd.DataFrame(outputs), pd.DataFrame(audits)

def extend_folds(frame, old_membership):
    """Preserve every old validation key; append eligible history only to training."""
    require(not old_membership.duplicated(KEYS + ['fold', 'role']).any(), 'Duplicate old fold keys')
    old_membership = old_membership.copy()
    old_membership['as_of_date'] = pd.to_datetime(old_membership.as_of_date)
    parts = [old_membership]
    boundary = frame.loc[frame.research_partition.eq('sealed_holdout'), 'as_of_date'].min()
    require(pd.notna(boundary), 'Missing sealed holdout boundary')
    hist = frame.loc[frame.source_scope.eq('history_2016_2019')]
    for fold, g in old_membership.groupby('fold'):
        valid = g.loc[g.role.eq('validation')]
        require(len(valid) > 0, 'Fold lacks validation')
        start = valid.as_of_date.min()
        admit = (
            hist.exact_supervision_candidate
            & hist.as_of_date.lt(start)
            & hist.target_end_date_12m.lt(start)
            & hist.supervision_available_date.lt(start)
        )
        added = hist.loc[admit, KEYS + ['observation_kind']].copy()
        added['fold'] = fold
        added['role'] = 'train'
        parts.append(added)
    membership = pd.concat(parts, ignore_index=True)
    require(not membership.duplicated(KEYS + ['fold', 'role']).any(), 'Duplicate expanded fold keys')
    audit = []
    for fold, g in membership.groupby('fold'):
        start = g.loc[g.role.eq('validation'), 'as_of_date'].min()
        require(not g.duplicated(KEYS).any(), 'Train/validation overlap')
        for role in ['train', 'validation']:
            keys = g.loc[g.role.eq(role), KEYS]
            rows = keys.merge(frame, on=KEYS, validate='one_to_one')
            require(len(rows) == len(keys), 'Missing fold rows')
            cutoff = start if role == 'train' else boundary
            require((rows.target_end_date_12m < cutoff).all() and (rows.supervision_available_date < cutoff).all(), 'Supervision crosses fold boundary')
            require(rows.feature_engineering_eligible.all(), 'Ineligible fold row')
            require(rows.observation_kind.isin(['exact', 'conditional_interval']).all(), 'Unsupported fold outcome')
            require(~rows.research_partition.eq('sealed_holdout').any(), 'Sealed holdout entered development')
            audit.append(
                dict(
                    fold=fold,
                    role=role,
                    rows=len(rows),
                    exact_rows=int(rows.observation_kind.eq('exact').sum()),
                    first_feature_date=rows.as_of_date.min(),
                    last_feature_date=rows.as_of_date.max(),
                    last_target_end=rows.target_end_date_12m.max(),
                )
            )
    return membership, pd.DataFrame(audit)

## 3. Base feature definitions inherited from v23/v24

These accounting ratios, growth formulas, and trailing price formulas construct the 64 base predictors on the `2016–2019` cohort so that the combined `2016–2026` base frame passes exact schema and value checks before the `v25` feature engine runs.

In [3]:
STATEMENT_FIELD_REGISTRY = {
    'income': ['revenue', 'costOfRevenue', 'grossProfit', 'operatingExpenses', 'operatingIncome', 'interestExpense', 'incomeBeforeTax', 'incomeTaxExpense', 'netIncome', 'ebit', 'ebitda', 'eps', 'epsDiluted', 'weightedAverageShsOut', 'weightedAverageShsOutDil'],
    'balance': ['cashAndCashEquivalents', 'shortTermInvestments', 'cashAndShortTermInvestments', 'netReceivables', 'inventory', 'totalCurrentAssets', 'totalAssets', 'accountPayables', 'totalCurrentLiabilities', 'shortTermDebt', 'longTermDebt', 'totalDebt', 'netDebt', 'totalLiabilities', 'retainedEarnings', 'totalStockholdersEquity', 'totalEquity'],
    'cashflow': ['netIncome', 'depreciationAndAmortization', 'stockBasedCompensation', 'changeInWorkingCapital', 'netCashProvidedByOperatingActivities', 'operatingCashFlow', 'capitalExpenditure', 'acquisitionsNet', 'netCashProvidedByInvestingActivities', 'netCashProvidedByFinancingActivities', 'commonDividendsPaid', 'freeCashFlow'],
}

INCOME_TTM_FIELDS = ['revenue', 'costOfRevenue', 'grossProfit', 'operatingExpenses', 'operatingIncome', 'interestExpense', 'incomeBeforeTax', 'incomeTaxExpense', 'netIncome', 'ebit', 'ebitda']
CASHFLOW_TTM_FIELDS = ['netIncome', 'depreciationAndAmortization', 'stockBasedCompensation', 'changeInWorkingCapital', 'netCashProvidedByOperatingActivities', 'operatingCashFlow', 'capitalExpenditure', 'acquisitionsNet', 'netCashProvidedByInvestingActivities', 'netCashProvidedByFinancingActivities', 'commonDividendsPaid', 'freeCashFlow']

TRADING_DAYS_PER_YEAR = 252
RETURN_HORIZONS = {'21d': 21, '63d': 63, '126d': 126, '252d': 252}
VOL_WINDOWS = {'63d': 63, '126d': 126, '252d': 252}

def build_security_price_features(group):
    g = group.sort_values('date').copy()
    stock_price = pd.to_numeric(g['stock_adj_close'], errors='coerce').where(lambda x: x > 0)
    market_price = pd.to_numeric(g['voo_adj_close'], errors='coerce').where(lambda x: x > 0)
    stock_log_price = np.log(stock_price)
    market_log_price = np.log(market_price)
    stock_log_return = stock_log_price.diff()
    market_log_return = market_log_price.diff()
    g['price_feature_source_date'] = g['date']
    g['price_observed_today'] = stock_price.notna()
    for label, horizon in RETURN_HORIZONS.items():
        stock_gross_return = stock_price / stock_price.shift(horizon)
        market_gross_return = market_price / market_price.shift(horizon)
        g[f'price_total_return_{label}'] = stock_gross_return - 1.0
        g[f'market_voo_return_{label}'] = market_gross_return - 1.0
        g[f'price_relative_return_voo_{label}'] = stock_gross_return / market_gross_return - 1.0
    stock_skipmonth_gross = stock_price.shift(21) / stock_price.shift(252)
    market_skipmonth_gross = market_price.shift(21) / market_price.shift(252)
    g['price_momentum_12m_ex_1m'] = stock_skipmonth_gross - 1.0
    g['price_relative_momentum_voo_12m_ex_1m'] = stock_skipmonth_gross / market_skipmonth_gross - 1.0
    for label, window in VOL_WINDOWS.items():
        g[f'price_realized_vol_{label}'] = stock_log_return.rolling(window=window, min_periods=window).std() * np.sqrt(TRADING_DAYS_PER_YEAR)
    downside_component = stock_log_return.clip(upper=0)
    g['price_downside_vol_252d'] = downside_component.pow(2).rolling(window=252, min_periods=252).mean().pow(0.5) * np.sqrt(TRADING_DAYS_PER_YEAR)
    rolling_peak_252 = stock_price.rolling(window=252, min_periods=252).max()
    g['price_drawdown_from_252d_high'] = stock_price / rolling_peak_252 - 1.0
    stock_market_cov_252 = stock_log_return.rolling(window=252, min_periods=252).cov(market_log_return)
    market_var_252 = market_log_return.rolling(window=252, min_periods=252).var()
    stock_var_252 = stock_log_return.rolling(window=252, min_periods=252).var()
    beta_252 = stock_market_cov_252 / market_var_252.where(market_var_252 > 0)
    corr_252 = stock_log_return.rolling(window=252, min_periods=252).corr(market_log_return).clip(lower=-1, upper=1)
    g['price_beta_voo_252d'] = beta_252
    g['price_correlation_voo_252d'] = corr_252
    idio_variance_252 = (stock_var_252 * (1.0 - corr_252.pow(2))).clip(lower=0)
    g['price_idiosyncratic_vol_252d'] = np.sqrt(idio_variance_252) * np.sqrt(TRADING_DAYS_PER_YEAR)
    g['market_voo_realized_vol_252d'] = market_log_return.rolling(window=252, min_periods=252).std() * np.sqrt(TRADING_DAYS_PER_YEAR)
    market_peak_252 = market_price.rolling(window=252, min_periods=252).max()
    g['market_voo_drawdown_from_252d_high'] = market_price / market_peak_252 - 1.0
    g['price_history_252d_complete'] = g['price_total_return_252d'].notna() & g['price_realized_vol_252d'].notna() & g['price_beta_voo_252d'].notna()
    return g

def fundamental_formulas(pit_fundamental_base_df):
    pit_fundamental_feature_df = pit_fundamental_base_df.copy()

    def numeric_series(df, col):
        return pd.to_numeric(df.get(col, pd.Series(np.nan, index=df.index)), errors='coerce').astype('float64')

    def safe_ratio(numerator, denominator, require_positive_denominator=False):
        numerator = pd.to_numeric(numerator, errors='coerce').astype('float64')
        denominator = pd.to_numeric(denominator, errors='coerce').astype('float64')
        valid = numerator.notna() & denominator.notna() & np.isfinite(numerator) & np.isfinite(denominator)
        if require_positive_denominator:
            valid &= denominator > 0
        else:
            valid &= denominator != 0
        result = pd.Series(np.nan, index=numerator.index, dtype='float64')
        result.loc[valid] = numerator.loc[valid] / denominator.loc[valid]
        return result

    revenue = numeric_series(pit_fundamental_feature_df, 'income__revenue_ttm')
    gross_profit = numeric_series(pit_fundamental_feature_df, 'income__grossProfit_ttm')
    operating_income = numeric_series(pit_fundamental_feature_df, 'income__operatingIncome_ttm')
    net_income = numeric_series(pit_fundamental_feature_df, 'income__netIncome_ttm')
    ebit = numeric_series(pit_fundamental_feature_df, 'income__ebit_ttm')
    ebitda = numeric_series(pit_fundamental_feature_df, 'income__ebitda_ttm')
    interest_expense = numeric_series(pit_fundamental_feature_df, 'income__interestExpense_ttm')
    ocf = numeric_series(pit_fundamental_feature_df, 'cashflow__operatingCashFlow_ttm')
    capex = numeric_series(pit_fundamental_feature_df, 'cashflow__capitalExpenditure_ttm')
    vendor_fcf = numeric_series(pit_fundamental_feature_df, 'cashflow__freeCashFlow_ttm')
    sbc = numeric_series(pit_fundamental_feature_df, 'cashflow__stockBasedCompensation_ttm')
    pit_fundamental_feature_df['fund_fcf_reconstructed_ttm'] = ocf + capex
    fcf_reconstructed = pit_fundamental_feature_df['fund_fcf_reconstructed_ttm']
    fcf_scale = pd.concat([vendor_fcf.abs(), fcf_reconstructed.abs(), pd.Series(1.0, index=pit_fundamental_feature_df.index)], axis=1).max(axis=1)
    pit_fundamental_feature_df['audit_vendor_fcf_relative_error'] = (vendor_fcf - fcf_reconstructed).abs() / fcf_scale
    pit_fundamental_feature_df['flag_vendor_fcf_definition_mismatch'] = pit_fundamental_feature_df['audit_vendor_fcf_relative_error'] > 0.01
    pit_fundamental_feature_df['flag_positive_capex_ttm'] = capex > 0
    pit_fundamental_feature_df['fund_capex_outflow_ttm'] = (-capex).where(capex <= 0)
    capex_outflow = pit_fundamental_feature_df['fund_capex_outflow_ttm']
    assets = numeric_series(pit_fundamental_feature_df, 'balance__totalAssets')
    assets_prior = numeric_series(pit_fundamental_feature_df, 'balance__totalAssets_prior_year')
    equity = numeric_series(pit_fundamental_feature_df, 'balance__totalStockholdersEquity')
    equity_prior = numeric_series(pit_fundamental_feature_df, 'balance__totalStockholdersEquity_prior_year')
    debt = numeric_series(pit_fundamental_feature_df, 'balance__totalDebt')
    net_debt = numeric_series(pit_fundamental_feature_df, 'balance__netDebt')
    cash = numeric_series(pit_fundamental_feature_df, 'balance__cashAndShortTermInvestments')
    current_assets = numeric_series(pit_fundamental_feature_df, 'balance__totalCurrentAssets')
    current_liabilities = numeric_series(pit_fundamental_feature_df, 'balance__totalCurrentLiabilities')
    average_assets = (assets + assets_prior) / 2.0
    average_assets = average_assets.where((assets > 0) & (assets_prior > 0))
    average_equity = (equity + equity_prior) / 2.0
    average_equity = average_equity.where(average_equity > 0)
    pit_fundamental_feature_df['fund_average_assets'] = average_assets
    pit_fundamental_feature_df['fund_average_equity'] = average_equity
    pit_fundamental_feature_df['fund_gross_margin_ttm'] = safe_ratio(gross_profit, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_operating_margin_ttm'] = safe_ratio(operating_income, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_ebitda_margin_ttm'] = safe_ratio(ebitda, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_net_margin_ttm'] = safe_ratio(net_income, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_ocf_margin_ttm'] = safe_ratio(ocf, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_fcf_margin_ttm'] = safe_ratio(fcf_reconstructed, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_roa_ttm'] = safe_ratio(net_income, average_assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_roe_ttm'] = safe_ratio(net_income, average_equity, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_asset_turnover_ttm'] = safe_ratio(revenue, average_assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_ocf_to_assets_ttm'] = safe_ratio(ocf, average_assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_accruals_to_assets_ttm'] = safe_ratio(net_income - ocf, average_assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_debt_to_assets'] = safe_ratio(debt, assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_net_debt_to_assets'] = safe_ratio(net_debt, assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_debt_to_equity'] = safe_ratio(debt, equity.where(equity > 0), require_positive_denominator=True)
    pit_fundamental_feature_df['fund_net_debt_to_ebitda_ttm'] = safe_ratio(net_debt, ebitda, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_current_ratio'] = safe_ratio(current_assets, current_liabilities, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_cash_to_assets'] = safe_ratio(cash, assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_working_capital_to_assets'] = safe_ratio(current_assets - current_liabilities, assets, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_interest_coverage_ttm'] = safe_ratio(ebit, interest_expense.abs(), require_positive_denominator=True)
    pit_fundamental_feature_df['fund_capex_intensity_ttm'] = safe_ratio(capex_outflow, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_sbc_to_revenue_ttm'] = safe_ratio(sbc, revenue, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_cash_conversion_ttm'] = safe_ratio(ocf, net_income, require_positive_denominator=True)

    def positive_base_growth(current, prior):
        current = pd.to_numeric(current, errors='coerce').astype('float64')
        prior = pd.to_numeric(prior, errors='coerce').astype('float64')
        valid = current.notna() & prior.notna() & np.isfinite(current) & np.isfinite(prior) & (prior > 0)
        result = pd.Series(np.nan, index=current.index, dtype='float64')
        result.loc[valid] = current.loc[valid] / prior.loc[valid] - 1.0
        return result

    revenue_prior = numeric_series(pit_fundamental_feature_df, 'income__revenue_ttm_prior_year')
    gross_profit_prior = numeric_series(pit_fundamental_feature_df, 'income__grossProfit_ttm_prior_year')
    operating_income_prior = numeric_series(pit_fundamental_feature_df, 'income__operatingIncome_ttm_prior_year')
    net_income_prior = numeric_series(pit_fundamental_feature_df, 'income__netIncome_ttm_prior_year')
    ebitda_prior = numeric_series(pit_fundamental_feature_df, 'income__ebitda_ttm_prior_year')
    ocf_prior = numeric_series(pit_fundamental_feature_df, 'cashflow__operatingCashFlow_ttm_prior_year')
    capex_prior = numeric_series(pit_fundamental_feature_df, 'cashflow__capitalExpenditure_ttm_prior_year')
    fcf_reconstructed_prior = ocf_prior + capex_prior
    pit_fundamental_feature_df['fund_fcf_reconstructed_ttm_prior_year'] = fcf_reconstructed_prior
    pit_fundamental_feature_df['fund_revenue_growth_yoy'] = positive_base_growth(revenue, revenue_prior)
    pit_fundamental_feature_df['fund_gross_profit_growth_yoy'] = positive_base_growth(gross_profit, gross_profit_prior)
    pit_fundamental_feature_df['fund_operating_income_growth_yoy'] = positive_base_growth(operating_income, operating_income_prior)
    pit_fundamental_feature_df['fund_ebitda_growth_yoy'] = positive_base_growth(ebitda, ebitda_prior)
    pit_fundamental_feature_df['fund_net_income_growth_yoy'] = positive_base_growth(net_income, net_income_prior)
    pit_fundamental_feature_df['fund_ocf_growth_yoy'] = positive_base_growth(ocf, ocf_prior)
    pit_fundamental_feature_df['fund_fcf_growth_yoy'] = positive_base_growth(fcf_reconstructed, fcf_reconstructed_prior)
    debt_prior = numeric_series(pit_fundamental_feature_df, 'balance__totalDebt_prior_year')
    cash_prior = numeric_series(pit_fundamental_feature_df, 'balance__cashAndShortTermInvestments_prior_year')
    pit_fundamental_feature_df['fund_assets_growth_yoy'] = positive_base_growth(assets, assets_prior)
    pit_fundamental_feature_df['fund_debt_growth_yoy'] = positive_base_growth(debt, debt_prior)
    pit_fundamental_feature_df['fund_cash_growth_yoy'] = positive_base_growth(cash, cash_prior)
    diluted_shares_current = numeric_series(pit_fundamental_feature_df, 'income__weightedAverageShsOutDil')
    diluted_shares_prior = numeric_series(pit_fundamental_feature_df, 'income__weightedAverageShsOutDil_prior_year_quarter')
    pit_fundamental_feature_df['fund_diluted_share_growth_yoy'] = positive_base_growth(diluted_shares_current, diluted_shares_prior)
    pit_fundamental_feature_df['fund_shareholder_yield_dilution_component'] = -pit_fundamental_feature_df['fund_diluted_share_growth_yoy']
    gross_margin_prior = safe_ratio(gross_profit_prior, revenue_prior, require_positive_denominator=True)
    operating_margin_prior = safe_ratio(operating_income_prior, revenue_prior, require_positive_denominator=True)
    net_margin_prior = safe_ratio(net_income_prior, revenue_prior, require_positive_denominator=True)
    fcf_margin_prior = safe_ratio(fcf_reconstructed_prior, revenue_prior, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_gross_margin_change_yoy'] = pit_fundamental_feature_df['fund_gross_margin_ttm'] - gross_margin_prior
    pit_fundamental_feature_df['fund_operating_margin_change_yoy'] = pit_fundamental_feature_df['fund_operating_margin_ttm'] - operating_margin_prior
    pit_fundamental_feature_df['fund_net_margin_change_yoy'] = pit_fundamental_feature_df['fund_net_margin_ttm'] - net_margin_prior
    pit_fundamental_feature_df['fund_fcf_margin_change_yoy'] = pit_fundamental_feature_df['fund_fcf_margin_ttm'] - fcf_margin_prior
    debt_to_assets_prior = safe_ratio(debt_prior, assets_prior, require_positive_denominator=True)
    cash_to_assets_prior = safe_ratio(cash_prior, assets_prior, require_positive_denominator=True)
    pit_fundamental_feature_df['fund_debt_to_assets_change_yoy'] = pit_fundamental_feature_df['fund_debt_to_assets'] - debt_to_assets_prior
    pit_fundamental_feature_df['fund_cash_to_assets_change_yoy'] = pit_fundamental_feature_df['fund_cash_to_assets'] - cash_to_assets_prior
    pit_fundamental_feature_df['flag_negative_equity'] = equity <= 0
    pit_fundamental_feature_df['flag_negative_ebitda_ttm'] = ebitda <= 0
    pit_fundamental_feature_df['flag_negative_fcf_ttm'] = fcf_reconstructed < 0
    pit_fundamental_feature_df['flag_new_debt_from_zero_yoy'] = (debt_prior == 0) & (debt > 0)
    return pit_fundamental_feature_df

## 4. Verify and load inputs

`N2_DIR` and `BASELINE_N3_DIR` artifacts are checked against their exported SHA-256 hashes. Baseline N3 must have an accepted handoff, the expected 64-feature schema, and precisely the preserved baseline N2 company-month keys.

In [4]:
# 1. Load and validate the completed v23 baseline contract and artifacts
baseline_contract_path = BASELINE_N3_DIR / 'pretraining_contract.json'
require(baseline_contract_path.is_file(), f'Missing baseline contract: {baseline_contract_path}')
baseline_contract = json.loads(baseline_contract_path.read_text())
require(
    baseline_contract.get('status') == 'TECHNICAL_HANDOFF_ACCEPTED_FOR_ARCHITECTURE_RESEARCH',
    'Baseline N3 handoff was not accepted',
)
FEATURE_COLUMNS = list(baseline_contract['feature_columns'])
require(len(FEATURE_COLUMNS) == 64 and len(set(FEATURE_COLUMNS)) == 64, 'Baseline N3 must have 64 unique features')

baseline_frame_path = BASELINE_N3_DIR / baseline_contract['frame_file']
baseline_fold_path = BASELINE_N3_DIR / baseline_contract['fold_membership_file']
for p in [baseline_frame_path, baseline_fold_path]:
    expected_sha = baseline_contract.get('artifact_sha256', {}).get(p.name)
    require(p.is_file() and (not expected_sha or sha_file(p) == expected_sha), f'Baseline artifact mismatch: {p}')

baseline_frame_df = pd.read_parquet(baseline_frame_path)
baseline_membership_df = pd.read_csv(baseline_fold_path, dtype={'security_key': 'string'})
for col in ['as_of_date', 'target_end_date_12m', 'supervision_available_date']:
    if col in baseline_frame_df.columns:
        baseline_frame_df[col] = pd.to_datetime(baseline_frame_df[col], errors='raise')
baseline_membership_df['as_of_date'] = pd.to_datetime(baseline_membership_df['as_of_date'], errors='raise')

# 2. Load and verify N2 historical tables and input_hashes
require(N2_DIR.is_dir(), f'Missing expanded N2 directory: {N2_DIR}')
require(VOO_PATH.is_file(), f'Missing VOO parquet: {VOO_PATH}')
voo_df = pd.read_parquet(VOO_PATH)

def discover_n2_tables(n2_dir: Path, baseline_df: pd.DataFrame):
    panel_df = ledger_df = prices_df = overlap_df = None
    tracked_files = [baseline_contract_path, baseline_frame_path, baseline_fold_path, VOO_PATH]
    for p in sorted(n2_dir.iterdir()):
        if not p.is_file() or p.suffix not in {'.parquet', '.csv', '.json'}:
            continue
        tracked_files.append(p)
        if p.suffix in {'.parquet', '.csv'}:
            df = pd.read_parquet(p) if p.suffix == '.parquet' else pd.read_csv(p)
            cols = set(df.columns)
            if {'constant_scale_diagnostic_pass', 'security_key'}.issubset(cols):
                overlap_df = df
            elif {'price_usable', 'adjClose', 'security_key', 'date'}.issubset(cols):
                prices_df = df
            elif {'ledger_row_id', 'statement_type', 'tradable_from', 'dependency_usable'}.issubset(cols):
                ledger_df = df
            elif {'security_key', 'as_of_date', 'fully_usable_source_state', 'income__ledger_row_id'}.issubset(cols):
                if len(df) == 24377:
                    panel_df = df
                elif panel_df is None:
                    panel_df = df
    require(all(x is not None for x in [panel_df, ledger_df, prices_df, overlap_df]), 'Could not locate all 4 N2 tables in N2_DIR')
    panel_df = panel_df.copy()
    panel_df['as_of_date'] = pd.to_datetime(panel_df['as_of_date'], errors='raise')
    # If N2 panel contains both cohorts (64,194 rows), verify baseline keys and slice 2016-2019 (24,377 rows)
    if len(panel_df) > 24377:
        base_keys = baseline_df[KEYS].sort_values(KEYS).reset_index(drop=True)
        n2_base = panel_df.loc[panel_df['as_of_date'] >= pd.Timestamp('2020-01-01'), KEYS].sort_values(KEYS).reset_index(drop=True)
        pd.testing.assert_frame_equal(n2_base, base_keys, check_dtype=False)
        panel_df = panel_df.loc[panel_df['as_of_date'] < pd.Timestamp('2020-01-01')].sort_values(KEYS).reset_index(drop=True)
    require(len(panel_df) == 24377, f'Expected 24,377 historical company-months, found {len(panel_df)}')
    return panel_df, ledger_df, prices_df, overlap_df, tracked_files

historical_panel_df, historical_ledger_df, historical_prices_df, price_overlap_df, tracked_files = discover_n2_tables(N2_DIR, baseline_frame_df)

# Preserve exact input_sha256 manifest from v24 if present and valid on disk; otherwise compute from tracked_files
v24_contract_path = PRIOR_V24_DIR / 'pretraining_contract.json'
if v24_contract_path.is_file():
    v24_contract = json.loads(v24_contract_path.read_text())
    candidate_hashes = v24_contract.get('input_sha256', {})
    if candidate_hashes and all(Path(p).is_file() and sha_file(p) == d for p, d in candidate_hashes.items()):
        input_hashes = dict(candidate_hashes)
    else:
        input_hashes = {str(p): sha_file(p) for p in tracked_files}
else:
    input_hashes = {str(p): sha_file(p) for p in tracked_files}

print('Verified input files:', len(input_hashes))
print('Historical panel rows:', len(historical_panel_df), '| Baseline v23 rows:', len(baseline_frame_df))

Verified input files: 17
Historical panel rows: 24377 | Baseline v23 rows: 39817


## 5. Engineer historical base features and outcomes (2016–2019)

All 24,377 historical company-months remain visible. Historical rows require complete core TTM inputs, a complete trailing annual price window, no unresolved overlap conflict, and no material balance identity discrepancy.

Reviewed `SIVB` liquidity exclusions are transferred by security and fiscal period. Positive-capex cases retain missing FCF-derived predictors pending sign resolution.

In [5]:
historical_financial_inputs_df, historical_dependency_df = build_financial_inputs(historical_panel_df, historical_ledger_df)
historical_fundamental_df = fundamental_formulas(historical_financial_inputs_df)

# Carry the reviewed SIVB historical liquidity exclusions across the new ledger IDs.
liquidity_mask = historical_fundamental_df.security_key.eq('SIVB') & pd.to_datetime(
    historical_fundamental_df.balance_period_end
).isin(pd.to_datetime(['2018-09-30', '2018-12-31', '2019-03-31', '2019-06-30', '2019-09-30', '2019-12-31']))

feature_mask_rows = []
for feature in ['fund_current_ratio', 'fund_working_capital_to_assets']:
    masked = historical_fundamental_df.loc[liquidity_mask, KEYS].copy()
    masked['feature'] = feature
    masked['reason'] = 'reviewed_SIVB_period_liquidity_definition_unverified'
    feature_mask_rows.append(masked)
    historical_fundamental_df.loc[liquidity_mask, feature] = np.nan

# Preserve missingness when a vendor capex sign conflicts with the inherited FCF formula.
positive_capex = historical_fundamental_df.flag_positive_capex_ttm.fillna(False)
for feature in [f for f in FEATURE_COLUMNS if f.startswith('fund_fcf_')]:
    masked = historical_fundamental_df.loc[positive_capex, KEYS].copy()
    masked['feature'] = feature
    masked['reason'] = 'positive_capex_sign_unresolved'
    feature_mask_rows.append(masked)
    historical_fundamental_df.loc[positive_capex, feature] = np.nan

historical_feature_masks_df = pd.concat(feature_mask_rows, ignore_index=True)
market_columns = [f for f in FEATURE_COLUMNS if f.startswith(('price_', 'market_voo_'))]
fund_columns = [f for f in FEATURE_COLUMNS if f.startswith('fund_')]
require(set(market_columns + fund_columns) == set(FEATURE_COLUMNS), 'Unknown inherited feature family')

historical_market_df, historical_price_audit_df = historical_market_and_labels(
    historical_panel_df, historical_prices_df, voo_df, price_overlap_df, market_columns
)

financial_metadata = ['fully_usable_source_state', 'income__ttm_valid', 'cashflow__ttm_valid']
core_ttm = [
    'income__revenue_ttm', 'income__operatingIncome_ttm', 'income__netIncome_ttm',
    'cashflow__operatingCashFlow_ttm', 'cashflow__capitalExpenditure_ttm', 'cashflow__freeCashFlow_ttm',
]
historical_fundamental_df['core_ttm_complete'] = historical_fundamental_df[core_ttm].notna().all(axis=1)

assets = historical_financial_inputs_df['balance__totalAssets']
liabilities = historical_financial_inputs_df['balance__totalLiabilities']
equity = historical_financial_inputs_df['balance__totalEquity'].combine_first(
    historical_financial_inputs_df['balance__totalStockholdersEquity']
)
scale = pd.concat([assets.abs(), (liabilities + equity).abs(), pd.Series(1.0, index=assets.index)], axis=1).max(axis=1)
relative_error = (assets - liabilities - equity).abs() / scale
historical_fundamental_df['balance_identity_relative_error'] = relative_error
historical_fundamental_df['balance_identity_conflict'] = relative_error.gt(0.01)

historical_frame_df = historical_fundamental_df[
    KEYS + fund_columns + financial_metadata + ['core_ttm_complete', 'balance_identity_relative_error', 'balance_identity_conflict']
].merge(historical_market_df, on=KEYS, validate='one_to_one')

f = historical_frame_df
f['feature_engineering_eligible'] = (
    f.fully_usable_source_state
    & f.income__ttm_valid
    & f.cashflow__ttm_valid
    & f.core_ttm_complete
    & f.price_history_complete
    & ~f.balance_identity_conflict
    & ~f.price_snapshot_status.eq('overlap_conflict_hold')
)
f['historical_exclusion_reasons'] = ''
for condition, reason in [
    (~f.fully_usable_source_state, 'N2_source_ineligible'),
    (~(f.income__ttm_valid & f.cashflow__ttm_valid & f.core_ttm_complete), 'TTM_missing_or_blocked'),
    (~f.price_history_complete, 'incomplete_trailing_price_history'),
    (f.balance_identity_conflict, 'balance_identity_conflict'),
    (f.price_snapshot_status.eq('overlap_conflict_hold'), 'price_snapshot_conflict'),
]:
    f.loc[condition, 'historical_exclusion_reasons'] += reason + ';'

f['source_scope'] = 'history_2016_2019'
f['research_partition'] = 'development_period'
f['exact_supervision_candidate'] = f.feature_engineering_eligible & f.observation_kind.eq('exact')
f['conditional_supervision_candidate'] = False
f['exact_loss_eligible'] = f.observation_kind.eq('exact')
f['conditional_loss_eligible'] = f.observation_kind.eq('exact')
f['loss_lower'] = f[TARGET]
f['loss_upper'] = f[TARGET]
f['label_available_date'] = f.supervision_available_date
f['label_status'] = np.where(f.observation_kind.eq('exact'), 'observed_historical_adjusted_price_path', f.historical_outcome_reason)

require(f.loc[~f.observation_kind.eq('exact'), TARGET].isna().all(), 'Unknown target was assigned a value')
require(np.isfinite(f.loc[f.observation_kind.eq('exact'), TARGET]).all(), 'Non-finite exact target')
require(not np.isinf(f[FEATURE_COLUMNS].to_numpy(dtype=float, na_value=np.nan)).any(), 'Infinite historical predictor')
print('Historical feature-eligible:', int(f.feature_engineering_eligible.sum()))
print('Historical exact and feature-eligible:', int(f.exact_supervision_candidate.sum()))
display(f.groupby([f.as_of_date.dt.year, 'feature_engineering_eligible', 'observation_kind']).size().rename('rows').reset_index())

Historical price histories processed: 100
Historical price histories processed: 200
Historical price histories processed: 300
Historical price histories processed: 400
Historical price histories processed: 500
Historical price histories processed: 600
Historical feature-eligible: 20301
Historical exact and feature-eligible: 20164


,as_of_date,feature_engineering_eligible,observation_kind,rows
0,2016,False,exact,388
1,2016,False,unknown,853
2,2016,True,exact,4816
3,2016,True,unknown,55
4,2017,False,exact,304
5,2017,False,unknown,761
6,2017,True,exact,4993
7,2017,True,unknown,37
8,2018,False,exact,257
9,2018,False,unknown,710


## 6. Combine cohorts, engineer v25 Scar-Tissue & Velocity features, and extend folds

First, the `2016–2019` historical frame and `2020+` baseline frame are combined and checked for 100% equality on all `baseline_v23` rows. Next, the `v25` feature engine computes path-dependent Ulcer Index, underwater duration, volatility asymmetry/term-structure, momentum curvature, 3m/6m fundamental velocity, and falling-knife interaction features across the unified panel.

In [6]:
# Idempotent: always pull the 64 base columns from baseline_contract so In [6] can be re-run safely
BASELINE_V24_FEATURE_COLUMNS = list(baseline_contract['feature_columns'])

baseline_with_scope = baseline_frame_df.copy()
baseline_with_scope['source_scope'] = 'baseline_v23'
pretraining_frame_df = (
    pd.concat(
        [historical_frame_df, baseline_with_scope],
        ignore_index=True,
        sort=False,
    )
    .sort_values(KEYS)
    .reset_index(drop=True)
)
for column in [
    'as_of_date',
    'target_end_date_12m',
    'supervision_available_date',
]:
    pretraining_frame_df[column] = pd.to_datetime(
        pretraining_frame_df[column], errors='raise'
    )
require(
    not pretraining_frame_df.duplicated(KEYS).any(),
    'Expanded frame has duplicate keys',
)

preserved = (
    pretraining_frame_df.loc[
        pretraining_frame_df.source_scope.eq('baseline_v23'),
        baseline_frame_df.columns,
    ]
    .sort_values(KEYS)
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(
    preserved,
    baseline_frame_df.sort_values(KEYS).reset_index(drop=True),
    check_dtype=False,
)


def safe_div_series(
    num: pd.Series, den: pd.Series, eps: float = 1e-6
) -> pd.Series:
    clean_den = pd.to_numeric(den, errors='coerce').where(
        lambda x: x.abs() > eps, np.nan
    )
    out = pd.to_numeric(num, errors='coerce') / clean_den
    return out.replace([np.inf, -np.inf], np.nan)


def engineer_v25_features_on_unified_panel(
    df: pd.DataFrame,
    base_cols: list[str],
    quarantine_macro_date_stamps: bool = True,
    quarantine_raw_vol_levels: bool = True,
) -> tuple[pd.DataFrame, list[str]]:
    df = df.sort_values(KEYS).reset_index(drop=True)

    # Guard causal 3m/6m lags against calendar gaps AND against the 42 'overlap_conflict_hold' securities
    g_sec = df.groupby('security_key', sort=False)
    days_3m = g_sec['as_of_date'].diff(3).dt.days
    days_6m = g_sec['as_of_date'].diff(6).dt.days
    conflict_hold = (
        df.get('price_snapshot_status', pd.Series('', index=df.index))
        .fillna('')
        .eq('overlap_conflict_hold')
    )
    conflict_in_window_6m = (
        conflict_hold.groupby(df['security_key'], sort=False)
        .transform(lambda s: s.rolling(7, min_periods=1).max())
        .astype(bool)
    )

    valid_3m = days_3m.between(75, 105) & (~conflict_in_window_6m)
    valid_6m = days_6m.between(165, 200) & (~conflict_in_window_6m)

    new_cols = {}

    # A. ULCER INDEX & CHRONIC UNDERWATER DURATION ("SCAR TISSUE")
    dd = df['price_drawdown_from_252d_high'].clip(upper=0.0)
    dd_sq = dd.pow(2)
    under_20 = (dd <= -0.20).astype('float64').where(dd.notna())
    under_40 = (dd <= -0.40).astype('float64').where(dd.notna())

    new_cols['price_ulcer_index_6m'] = np.sqrt(
        dd_sq.groupby(df['security_key'], sort=False).transform(
            lambda s: s.rolling(6, min_periods=3).mean()
        )
    )
    new_cols['price_ulcer_index_12m'] = np.sqrt(
        dd_sq.groupby(df['security_key'], sort=False).transform(
            lambda s: s.rolling(12, min_periods=6).mean()
        )
    )
    new_cols['price_underwater_duration_20pct_12m'] = under_20.groupby(
        df['security_key'], sort=False
    ).transform(lambda s: s.rolling(12, min_periods=6).mean())
    new_cols['price_underwater_duration_40pct_12m'] = under_40.groupby(
        df['security_key'], sort=False
    ).transform(lambda s: s.rolling(12, min_periods=6).mean())
    new_cols['price_acute_vs_chronic_drawdown'] = safe_div_series(
        dd.abs(), new_cols['price_ulcer_index_12m'] + 0.01
    )
    new_cols['price_drawdown_velocity_3m'] = (
        g_sec['price_drawdown_from_252d_high'].diff(3).where(valid_3m, np.nan)
    )
    new_cols['price_excess_drawdown_vs_voo_252d'] = (
        df['price_drawdown_from_252d_high']
        - df['market_voo_drawdown_from_252d_high']
    )

    # B. VOLATILITY ASYMMETRY & TERM STRUCTURE
    new_cols['price_downside_vol_ratio_252d'] = safe_div_series(
        df['price_downside_vol_252d'], df['price_realized_vol_252d']
    )
    new_cols['price_idio_vol_ratio_252d'] = safe_div_series(
        df['price_idiosyncratic_vol_252d'], df['price_realized_vol_252d']
    )
    new_cols['price_vol_term_structure_63d_252d'] = (
        safe_div_series(
            df['price_realized_vol_63d'], df['price_realized_vol_252d']
        )
        - 1.0
    )
    new_cols['price_vol_term_structure_126d_252d'] = (
        safe_div_series(
            df['price_realized_vol_126d'], df['price_realized_vol_252d']
        )
        - 1.0
    )
    new_cols['price_rel_vol_to_voo_252d'] = safe_div_series(
        df['price_realized_vol_252d'], df['market_voo_realized_vol_252d']
    )

    # C. MULTI-HORIZON MOMENTUM RATIOS & 2ND-DERIVATIVE PRICE CURVATURE
    r21 = df['price_total_return_21d']
    r63 = df['price_total_return_63d']
    r126 = df['price_total_return_126d']
    r252 = df['price_total_return_252d']
    rel63 = df['price_relative_return_voo_63d']
    rel252 = df['price_relative_return_voo_252d']

    new_cols['price_momentum_12m_ex_3m'] = (
        safe_div_series(1.0 + r252, 1.0 + r63) - 1.0
    )
    new_cols['price_relative_momentum_voo_12m_ex_3m'] = (
        safe_div_series(1.0 + rel252, 1.0 + rel63) - 1.0
    )

    pace_21d = ((1.0 + r252.clip(lower=-0.99)).pow(21.0 / 252.0)) - 1.0
    pace_63d = ((1.0 + r252.clip(lower=-0.99)).pow(63.0 / 252.0)) - 1.0
    new_cols['price_return_curvature_21d_vs_252d'] = r21 - pace_21d
    new_cols['price_return_curvature_63d_vs_252d'] = r63 - pace_63d

    prior_63d = safe_div_series(1.0 + r126, 1.0 + r63) - 1.0
    new_cols['price_quarterly_return_accel_63d_vs_126d'] = r63 - prior_63d
    new_cols['price_martin_ratio_252d'] = safe_div_series(
        rel252, new_cols['price_ulcer_index_12m'] + 0.02
    )

    # D. SECOND-DERIVATIVE FUNDAMENTAL VELOCITY (QoQ = 3m lag, Semi-Annual = 6m lag)
    yoy_velocity_targets = [
        'fund_revenue_growth_yoy',
        'fund_gross_profit_growth_yoy',
        'fund_operating_income_growth_yoy',
        'fund_ebitda_growth_yoy',
        'fund_net_income_growth_yoy',
        'fund_ocf_growth_yoy',
        'fund_fcf_growth_yoy',
        'fund_cash_to_assets_change_yoy',
        'fund_debt_to_assets_change_yoy',
    ]
    for col in yoy_velocity_targets:
        if col in df.columns:
            new_cols[f'{col}_qoq_accel_3m'] = (
                g_sec[col].diff(3).where(valid_3m, np.nan)
            )
            new_cols[f'{col}_semiannual_accel_6m'] = (
                g_sec[col].diff(6).where(valid_6m, np.nan)
            )

    ttm_trajectory_targets = [
        'fund_gross_margin_ttm',
        'fund_operating_margin_ttm',
        'fund_fcf_margin_ttm',
        'fund_roa_ttm',
        'fund_roe_ttm',
        'fund_cash_conversion_ttm',
        'fund_accruals_to_assets_ttm',
        'fund_debt_to_assets',
        'fund_cash_to_assets',
        'fund_interest_coverage_ttm',
    ]
    for col in ttm_trajectory_targets:
        if col in df.columns:
            new_cols[f'{col}_qoq_delta_3m'] = (
                g_sec[col].diff(3).where(valid_3m, np.nan)
            )

    new_cols['fund_ocf_vs_netinc_growth_spread'] = (
        df['fund_ocf_growth_yoy'] - df['fund_net_income_growth_yoy']
    )
    new_cols['fund_fcf_vs_rev_growth_spread'] = (
        df['fund_fcf_growth_yoy'] - df['fund_revenue_growth_yoy']
    )
    new_cols['fund_debt_vs_cash_growth_spread'] = (
        df['fund_debt_growth_yoy'] - df['fund_cash_growth_yoy']
    )

    # E. CROSS-DOMAIN "FALLING KNIFE" VS. "COILED SPRING" INTERACTIONS
    dd_mag = dd.abs()
    new_cols['fund_drawdown_x_cash_burn'] = (
        dd_mag * df['fund_cash_to_assets_change_yoy']
    )
    new_cols['fund_drawdown_x_ocf_accel'] = dd_mag * new_cols[
        'fund_ocf_growth_yoy_qoq_accel_3m'
    ].clip(-2.0, 2.0)
    new_cols['fund_drawdown_x_margin_trajectory'] = (
        dd_mag * df['fund_operating_margin_change_yoy']
    )
    new_cols['fund_drawdown_x_leverage_stress'] = dd * df['fund_debt_to_assets']
    new_cols['price_drawdown_x_63d_stabilization'] = dd_mag * r63
    new_cols['price_drawdown_x_21d_stabilization'] = dd_mag * r21

    engineered_block = pd.DataFrame(
        new_cols, index=df.index, dtype='float64'
    ).replace([np.inf, -np.inf], np.nan)
    df = pd.concat([df, engineered_block], axis=1)

    updated_cols = list(base_cols) + list(engineered_block.columns)

    if quarantine_macro_date_stamps:
        updated_cols = [
            c for c in updated_cols if not c.startswith('market_voo_')
        ]

    if quarantine_raw_vol_levels:
        raw_vol_cols = {
            'price_realized_vol_63d',
            'price_realized_vol_126d',
            'price_realized_vol_252d',
            'price_downside_vol_252d',
        }
        updated_cols = [c for c in updated_cols if c not in raw_vol_cols]

    require(
        len(updated_cols) == len(set(updated_cols)),
        'Duplicate predictor names in v25 whitelist',
    )
    require(
        all(c.startswith(('fund_', 'price_', 'market_')) for c in updated_cols),
        'Every predictor must start with fund_, price_, or market_ for Notebook 04',
    )
    return df, updated_cols


pretraining_frame_df, FEATURE_COLUMNS = engineer_v25_features_on_unified_panel(
    pretraining_frame_df,
    base_cols=BASELINE_V24_FEATURE_COLUMNS,
    quarantine_macro_date_stamps=QUARANTINE_MACRO_VOO_DATE_STAMPS,
    quarantine_raw_vol_levels=QUARANTINE_RAW_VOLATILITY_LEVELS,
)

# Align security_key dtype across frames so pd.concat inside extend_folds preserves string dtype
pretraining_frame_df['security_key'] = pretraining_frame_df[
    'security_key'
].astype('string')
baseline_membership_df['security_key'] = baseline_membership_df[
    'security_key'
].astype('string')

pretraining_fold_membership_df, fold_summary_df = extend_folds(
    pretraining_frame_df, baseline_membership_df
)
old_validation = (
    baseline_membership_df.loc[
        baseline_membership_df.role.eq('validation'), ['fold'] + KEYS
    ]
    .sort_values(['fold'] + KEYS)
    .reset_index(drop=True)
)
new_validation = (
    pretraining_fold_membership_df.loc[
        pretraining_fold_membership_df.role.eq('validation'), ['fold'] + KEYS
    ]
    .sort_values(['fold'] + KEYS)
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(
    old_validation, new_validation, check_dtype=False
)
require(
    not np.isinf(
        pretraining_frame_df[FEATURE_COLUMNS].to_numpy(
            dtype=float, na_value=np.nan
        )
    ).any(),
    'Infinite predictor in expanded frame',
)
require(
    historical_frame_df.exact_supervision_candidate.any(),
    'No new eligible exact history; inspect exclusions before proceeding',
)

checks = {
    'baseline_values_preserved': True,
    'feature_whitelist_preserved': True,
    'unique_keys': True,
    'all_historical_dependencies_available_as_of': bool(
        (
            historical_dependency_df.tradable_from
            <= historical_dependency_df.as_of_date
        ).all()
    ),
    'unknown_labels_not_imputed': True,
    'unchanged_validation_keys': True,
    'purged_training_targets': True,
    'sealed_holdout_preserved': True,
    'historical_price_snapshots_not_spliced': True,
    'new_eligible_history_present': True,
    'v25_scar_tissue_and_velocity_engineered': True,
}

counts_df = (
    pretraining_frame_df.groupby(
        [
            pretraining_frame_df.as_of_date.dt.year.rename('year'),
            'source_scope',
            'feature_engineering_eligible',
            'observation_kind',
        ],
        dropna=False,
    )
    .size()
    .rename('rows')
    .reset_index()
)

eligible_hist_mask = pretraining_frame_df.source_scope.eq(
    'history_2016_2019'
) & pretraining_frame_df.feature_engineering_eligible.fillna(False)
eligible_base_mask = pretraining_frame_df.source_scope.eq(
    'baseline_v23'
) & pretraining_frame_df.feature_engineering_eligible.fillna(False)
coverage_df = pd.DataFrame(
    {
        'feature': FEATURE_COLUMNS,
        'historical_nonmissing_fraction': (
            pretraining_frame_df.loc[eligible_hist_mask, FEATURE_COLUMNS]
            .notna()
            .mean()
            .reindex(FEATURE_COLUMNS)
            .to_numpy()
        ),
        'baseline_nonmissing_fraction': (
            pretraining_frame_df.loc[eligible_base_mask, FEATURE_COLUMNS]
            .notna()
            .mean()
            .reindex(FEATURE_COLUMNS)
            .to_numpy()
        ),
    }
)
print('Combined company-months:', len(pretraining_frame_df))
print(
    f'v24 base features ({len(BASELINE_V24_FEATURE_COLUMNS)}) -> v25 active features ({len(FEATURE_COLUMNS)})'
)
display(fold_summary_df)
display(coverage_df.tail(25))

Combined company-months: 64194
v24 base features (64) -> v25 active features (109)


,fold,role,rows,exact_rows,first_feature_date,last_feature_date,last_target_end
0,0,train,29034,29034,2016-01-29,2021-06-30,2022-06-30
1,0,validation,2969,2964,2022-07-29,2022-12-30,2023-12-29
2,1,train,32007,32006,2016-01-29,2021-12-31,2022-12-30
3,1,validation,2942,2942,2023-01-31,2023-06-30,2024-06-28


,feature,historical_nonmissing_fraction,baseline_nonmissing_fraction
84,fund_fcf_growth_yoy_qoq_accel_3m,0.791784,0.843142
85,fund_fcf_growth_yoy_semiannual_accel_6m,0.724004,0.824518
86,fund_cash_to_assets_change_yoy_qoq_accel_3m,0.926654,0.979474
87,fund_cash_to_assets_change_yoy_semiannual_acce...,0.858529,0.968890
88,fund_debt_to_assets_change_yoy_qoq_accel_3m,0.926654,0.979474
89,fund_debt_to_assets_change_yoy_semiannual_acce...,0.858529,0.968890
90,fund_gross_margin_ttm_qoq_delta_3m,0.927935,0.983122
91,fund_operating_margin_ttm_qoq_delta_3m,0.927935,0.983122
92,fund_fcf_margin_ttm_qoq_delta_3m,0.924339,0.982814
93,fund_roa_ttm_qoq_delta_3m,0.925669,0.978473


## 7. Export the documented v25 pretraining handoff

Creates a fresh `data/interim/audits/nb03_history_v25/<RUN_ID>` directory, verifies all SHA-256 checksums, writes `pretraining_contract.json`, `pretraining_frame.parquet`, and `pretraining_fold_membership.csv`, and prints the new `ARTIFACT_DIR` path for `Notebook 04`.

In [7]:
RUN_ID = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = PROJECT_ROOT / 'data/interim/audits/nb03_history_v25' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

pretraining_frame_df.to_parquet(RUN_DIR / 'pretraining_frame.parquet', index=False)
pretraining_fold_membership_df.to_csv(RUN_DIR / 'pretraining_fold_membership.csv', index=False)
baseline_membership_df.to_csv(RUN_DIR / 'baseline_control_fold_membership.csv', index=False)
historical_dependency_df.to_parquet(RUN_DIR / 'historical_financial_dependencies.parquet', index=False)
historical_frame_df.to_parquet(RUN_DIR / 'historical_pretraining_frame.parquet', index=False)

for name, df in [
    ('observation_counts', counts_df),
    ('fold_summary', fold_summary_df),
    ('historical_feature_coverage', coverage_df),
    ('historical_feature_masks', historical_feature_masks_df),
    ('historical_price_audit', historical_price_audit_df),
]:
    df.to_csv(RUN_DIR / (name + '.csv'), index=False)

historical_frame_df.loc[
    ~historical_frame_df.exact_supervision_candidate,
    KEYS + ['feature_engineering_eligible', 'observation_kind', 'historical_exclusion_reasons', 'historical_outcome_reason'],
].to_csv(RUN_DIR / 'historical_exclusions.csv', index=False)

limitations = list(baseline_contract.get('limitations', [])) + [
    'Historical vendor values may reflect subsequent restatements; original-value vintages remain uncertified.',
    'Historical membership and many issuer bindings are provisional. CIK and continuity screens are not complete security-level certification.',
    'Historical outcomes use complete observed adjusted-price paths. New historical delisting/cash/stock consideration cases are unknown unless a valid continuous series exists; no zero/midpoint imputation.',
    'The 42 failed overlap diagnostics are conservatively held from historical training. Independent historical series without overlap remain separately identified.',
    'Historical and baseline price snapshots are never spliced. Baseline features and labels are preserved, so a snapshot-vintage difference remains across cohorts.',
    'Historical TTM resolves versions as of each observation; baseline v23 values remain frozen. This is an incremental extension, not a full vintage reconstruction.',
    'Historical eligibility additionally requires complete trailing prices and passes the balance identity screen; report exclusions when comparing cohorts.',
    'Observations have overlapping one-year outcomes; row counts are not independent sample counts. Validation and sealed holdout dates are unchanged.',
    'Missing-data selection bias remains possible, especially for historical exits. Models cannot recover unobserved outcomes.',
    'v25 adds causal path-dependent Ulcer Index, underwater duration, volatility term-structure/asymmetry, momentum curvature, and 3m/6m fundamental velocity features while quarantining raw macro VOO date-stamp columns and raw volatility levels.',
]

contract = {
    **baseline_contract,
    'revision': VERSION,
    'status': 'TECHNICAL_HANDOFF_ACCEPTED_FOR_ARCHITECTURE_RESEARCH',
    'technical_checks': checks,
    'rows': len(pretraining_frame_df),
    'feature_columns': FEATURE_COLUMNS,
    'feature_count': len(FEATURE_COLUMNS),
    'baseline_v24_feature_columns': BASELINE_V24_FEATURE_COLUMNS,
    'training_authorized': False,
    'models_trained': False,
    'source_certification_complete': False,
    'frame_file': 'pretraining_frame.parquet',
    'fold_membership_file': 'pretraining_fold_membership.csv',
    'baseline_control_fold_membership_file': 'baseline_control_fold_membership.csv',
    'source_fingerprint': hashlib.sha256(json.dumps(input_hashes, sort_keys=True).encode()).hexdigest(),
    'input_sha256': input_hashes,
    'limitations': limitations,
    'exact_baseline_rule': 'Join the explicit fold keys and select observation_kind == exact; use only feature_columns as X.',
    'labels_imputed_by_v24': 0,
    'baseline_values_modified_by_v24': 0,
    'historical_exact_eligible_rows': int(historical_frame_df.exact_supervision_candidate.sum()),
    'historical_company_months': len(historical_frame_df),
    'baseline_n3_dir': str(BASELINE_N3_DIR),
    'expanded_n2_dir': str(N2_DIR),
}

for path, digest in input_hashes.items():
    require(sha_file(path) == digest, 'Input changed during execution: ' + path)

contract['artifact_sha256'] = {p.name: sha_file(p) for p in RUN_DIR.iterdir() if p.is_file()}
write_json(contract, RUN_DIR / 'pretraining_contract.json')
(RUN_DIR / 'pretraining_limitations.txt').write_text('\n'.join('- ' + s for s in limitations))

review = {
    k: v
    for k, v in contract.items()
    if k not in ['input_sha256', 'artifact_sha256', 'feature_columns', 'baseline_v24_feature_columns']
}
write_json(review, RUN_DIR / 'handoff_report.json')
review_files = [p for p in RUN_DIR.iterdir() if p.suffix in ['.csv', '.json', '.txt']]
write_json({p.name: sha_file(p) for p in review_files}, RUN_DIR / 'review_sha256.json')
review_files.append(RUN_DIR / 'review_sha256.json')

bundle_path = RUN_DIR / f'review_bundle_{VERSION}_{RUN_ID}.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in review_files:
        z.write(p, p.name)

print(contract['status'])
print('N4 ARTIFACT_DIR:', RUN_DIR)
print('Review ZIP:', bundle_path)

TECHNICAL_HANDOFF_ACCEPTED_FOR_ARCHITECTURE_RESEARCH
N4 ARTIFACT_DIR: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb03_history_v25/20261030T141016323170Z
Review ZIP: /Users/neilyejjey/stock_signal_engine_v2/data/interim/audits/nb03_history_v25/20261030T141016323170Z/review_bundle_nb03_v25_SCAR_TISSUE_VELOCITY_20261030T141016323170Z.zip


In [8]:
import json
from pathlib import Path


def generate_nb02_v4_certified():
    project_root = Path("/Users/neilyejjey/stock_signal_engine_v2")
    nb_dir = (
        project_root / "Notebooks"
        if (project_root / "Notebooks").is_dir()
        else Path.cwd()
    )
    out_path = nb_dir / "02_expanded_historical_pit_source_panel_v4_CERTIFIED.ipynb"

    def md_cell(cell_id, text):
        lines = text.strip().splitlines()
        return {
            "cell_type": "markdown",
            "id": cell_id,
            "metadata": {},
            "source": [line + "\n" for line in lines[:-1]]
            + ([lines[-1]] if lines else []),
        }

    def code_cell(cell_id, code):
        lines = code.strip("\n").splitlines()
        return {
            "cell_type": "code",
            "execution_count": None,
            "id": cell_id,
            "metadata": {},
            "outputs": [],
            "source": [line + "\n" for line in lines[:-1]]
            + ([lines[-1]] if lines else []),
        }

    cells = [
        md_cell(
            "nb02-v4-000",
            """# Notebook 02 v4 — Certified Expanded Point-in-Time historical Source Panel

**Sequential Pipeline Overhaul — Step 2 of 5 (`Notebook 02`).**
This notebook integrates our Step 1 Certified Universe (`universe_security_master_v5_CERTIFIED.parquet` and `candidate_membership_v5_CERTIFIED.parquet`) to construct the final Point-in-Time (PIT) source panel for January 2016–December 2019, maintaining strict parity and preserving the repaired 2020+ snapshot.

### Core Enhancements:
1. **Verifiable Data Lineage:** Imports historical metadata from the `Notebook 01 v5` verified run directories.
2. **Dynamic Identity & CIK Alignments:** Incorporates the `admissible_for_nb02_history_2016_2019` flag and mapped CIK IDs, resolving dual-class share alignments using `canonical_issuer_id` instead of 8 manually-hardcoded overrides.
3. **Advanced point-In-Time Availability Verification:** Applies trading-session sequence filters on raw balance-sheet, income-statement, and cash-flow data. This maps filed information to the `tradable_from` next-market session based strictly on calendar constraints (ensuring publication availability date is less than the session datetime, checking standard weekend or holiday gaps).
4. **Incorporation of Terminal Exit Chronologies:** Embeds exit classifications and exact terminal dates to eradicate future leakages.
5. **Robust Handoff Checkpoint Verification:** Operates 8 executable point-in-time checks, outputting `expanded_n2_contract.json` and a diagnostic visual bundle.
""",
        ),
        code_cell(
            "nb02-v4-001",
            """from pathlib import Path
import hashlib
import json
import os
import shutil
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

VERSION = 'nb02_v4_HISTORY_2016_CERTIFIED'
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name.lower() == 'notebooks' else HERE
DATA = PROJECT_ROOT / 'data'

# Search for the latest Notebook 01 v5 Certified runs
v5_audits_dir = DATA / 'interim/audits/nb01_history_v5_certified'
if not v5_audits_dir.is_dir():
    raise FileNotFoundError(f"Step 1 certified directory not found at: {v5_audits_dir}. Please run Notebook 01 v5 first.")
v5_runs = sorted([p for p in v5_audits_dir.iterdir() if p.is_dir() and (p / 'nb01_universe_certification_contract.json').exists()])
if not v5_runs:
    raise FileNotFoundError("Could not find any completed Step 1 Certified Runs.")
CERTIFIED_N1_DIR = v5_runs[-1]

# Baseline notebook 2 runs (N2)
BASELINE_DIR = DATA / 'interim/audits/nb02_v2/20261014T163316Z'
CACHE_ROOTS = [
    DATA / 'raw/history_expansion/historical_expansion_v2_20261019',
    DATA / 'raw/history_expansion/historical_reconciliation_v3',
    DATA / 'raw/history_expansion/historical_issuer_recovery_v4',
]
CALENDAR_PATH = DATA / 'raw/market/VOO_dividend_adjusted_2010_plus.parquet'
BASELINE_PRICE_PATH = DATA / 'interim/panel/equity_daily_prices_2018_plus.parquet'

MAX_STATEMENT_AGE_DAYS = 180
ACCEPT_PROVISIONAL_VENDOR_HISTORY = True

PINNED_HASHES = {
    'baseline_panel': '846de7d0ca2e06908940b23786bbbb21b5ffa8f925d3f48f05caa620c0773a47',
    'baseline_ledger': '183f268ec243d4d859973cefd08b04fbf0a38366e5e1593314df2f66a6d213c3',
}

INPUT_PATHS = {
    'baseline_panel': BASELINE_DIR / 'candidate_pit_source_panel.parquet',
    'baseline_ledger': BASELINE_DIR / 'candidate_statement_ledger.parquet',
    'certified_membership': CERTIFIED_N1_DIR / 'candidate_membership_v5_CERTIFIED.parquet',
    'certified_master': CERTIFIED_N1_DIR / 'universe_security_master_v5_CERTIFIED.parquet',
    'calendar': CALENDAR_PATH,
    'baseline_prices': BASELINE_PRICE_PATH,
}

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

print('Verified latest Step 1 Certified Directory:', CERTIFIED_N1_DIR.name)
print('Project ROOT:', PROJECT_ROOT)""",
        ),
        md_cell(
            "nb02-v4-002",
            """## 2. Reusable point-In-Time Pipeline Methods

Builds historical financial and EOD pricing grids utilizing next-session calendar and data timing constraints to avoid future observation overlaps.""",
        ),
        code_cell(
            "nb02-v4-003",
            """KINDS = {
    'income-statement': 'income',
    'balance-sheet-statement': 'balance',
    'cash-flow-statement': 'cashflow'
}
PRICE_ENDPOINT = 'historical-price-eod/dividend-adjusted'

def canonical(x):
    return json.dumps(x, sort_keys=True, separators=(',', ':'), default=str)

def dump(x, path):
    Path(path).write_text(json.dumps(x, indent=2, default=str), encoding='utf-8')

def day(x):
    try:
        t = pd.Timestamp(x)
        if pd.isna(t): return pd.NaT
        if t.tzinfo is not None: t = t.tz_convert('America/New_York').tz_localize(None)
        return t.normalize()
    except (TypeError, ValueError): return pd.NaT

def read_caches(roots, symbols):
    seen, financials, prices, audit = {}, [], [], []
    for root in roots:
        if not root.is_dir(): continue
        files = sorted(root.glob('*.json'))
        print(f'Reading {len(files):,} cached responses from {root.name}')
        for path in files:
            e = json.loads(path.read_text())
            ep, params = e.get('endpoint'), e.get('params', {})
            symbol = params.get('symbol')
            if symbol not in symbols or ep not in {*KINDS, PRICE_ENDPOINT}: continue
            if ep in KINDS and params.get('period') != 'quarter': continue
            data = e.get('data')
            rid = hashlib.sha256(canonical({'endpoint': ep, 'params': params}).encode()).hexdigest()
            if path.stem != rid or not isinstance(data, list) or not all(isinstance(r, dict) for r in data):
                raise ValueError(f'Cache schema/request mismatch: {path}')
            digest = hashlib.sha256(canonical(data).encode()).hexdigest()
            if digest != e.get('payload_sha256'): raise ValueError(f'Cache checksum mismatch: {path}')
            state = 'selected' if rid not in seen else ('duplicate' if seen[rid] == digest else 'alternate_snapshot_not_used')
            audit.append(dict(path=str(path), request_id=rid, payload_sha256=digest, endpoint=ep,
                              query_symbol=symbol, rows=len(data), state=state))
            if rid in seen: continue
            seen[rid] = digest
            for number, record in enumerate(data):
                date_text = str(record.get('date', ''))[:10]
                if ep in KINDS and not ('2012-01-01' <= date_text <= '2019-12-31'): continue
                if ep == PRICE_ENDPOINT and not ('2014-01-01' <= date_text <= '2021-12-31'): continue
                r = dict(record) if ep in KINDS else {k: record.get(k) for k in ['date', 'symbol', 'adjClose']}
                r.update(security_key=symbol, source_request_id=rid, source_record_number=number,
                         source_payload_sha256=digest, source_record_sha256=hashlib.sha256(canonical(record).encode()).hexdigest())
                if ep in KINDS:
                    r['statement_type'] = KINDS[ep]
                    financials.append(r)
                else:
                    r['request_from'] = params.get('from')
                    r['request_to'] = params.get('to')
                    prices.append(r)
    if not financials or not prices: raise ValueError('No financial/price records found in configured caches')
    return pd.DataFrame(financials), pd.DataFrame(prices), pd.DataFrame(audit)

def monthly_base(membership, sessions):
    m = membership.copy()
    for c in ['member_from', 'member_to']: m[c] = pd.to_datetime(m[c], errors='raise')
    dates = pd.Series(sessions).groupby(sessions.to_period('M')).max()
    rows = []
    for d in dates.loc[dates.between('2016-01-01', '2019-12-31')]:
        active = m.loc[m.member_from.le(d) & (m.member_to.isna() | m.member_to.gt(d))]
        for r in active.to_dict('records'):
            rows.append(dict(security_key=r['candidate_security_key'], as_of_date=d,
                             query_symbol=r['query_symbol'], fmp_membership_from=r['member_from'],
                             fmp_membership_to=r['member_to'], source_scope='history_2016_2019',
                             membership_status='reconstructed_vendor_history_with_scoped_repairs',
                             canonical_issuer_id=r['canonical_issuer_id'],
                             is_dual_class_member=r['is_dual_class_member'],
                             identity_certification_status=r['identity_certification_status'],
                             admissible_for_nb02_history_2016_2019=r['admissible_for_nb02_history_2016_2019'],
                             exit_classification=r['exit_classification'],
                             documented_delisted_date=pd.to_datetime(r['documented_delisted_date'], errors='coerce')))
    base = pd.DataFrame(rows)
    if base.empty or base.duplicated(['security_key', 'as_of_date']).any(): raise ValueError('Invalid monthly universe')
    if base.as_of_date.nunique() != 48: raise ValueError('Expected 48 observed month ends for 2016-2019')
    return base

def prepare_ledger(raw, sessions, security_master):
    l = raw.copy()
    l['date'] = l['date'].map(day)
    l = l.loc[l.date.between('2012-01-01', '2019-12-31')].copy()
    for c in ['acceptedDate', 'filingDate', 'fiscalYear', 'period', 'cik', 'symbol']:
        if c not in l: l[c] = None

    l = l.merge(security_master[[
        'candidate_security_key',
        'resolved_historical_cik',
        'canonical_issuer_id',
        'identity_certification_status',
        'exit_classification',
        'documented_delisted_date',
        'price_coverage_status',
    ]], left_on='security_key', right_on='candidate_security_key', how='left')

    l['normalized_cik'] = l['resolved_historical_cik'].fillna('')
    l['identity_status'] = l['identity_certification_status'].fillna('missing_cik')
    l['identity_usable'] = ~l['identity_status'].isin([
        'REORG_TICKER_COLLISION_QUARANTINED',
        'KNOWN_ISSUER_BINDING_HOLD_QUARANTINED',
        'MULTIPLE_HISTORICAL_CIKS_CONFLICT',
        'RESPONSE_SYMBOL_MISMATCH',
        'MISSING_CIK_UNRESOLVED'
    ])

    l['availability_date'] = pd.NaT
    l['tradable_from'] = pd.NaT
    l['timing_status'] = 'no_valid_publication_date'

    for idx, r in l.iterrows():
        dates = [day(r.get(c)) for c in ['acceptedDate', 'filingDate']]
        valid = [d for d in dates if pd.notna(d) and d > r.date]
        if not valid: continue
        available = max(valid)
        pos = sessions.searchsorted(available, side='right')
        l.at[idx, 'availability_date'] = available
        if available < sessions.min() or pos >= len(sessions): continue
        l.at[idx, 'tradable_from'] = sessions[pos]
        l.at[idx, 'timing_status'] = 'later_valid_vendor_date_next_session'

    l['quarter_usable'] = l.period.isin(['Q1', 'Q2', 'Q3', 'Q4'])
    l['pit_usable_timestamp'] = l.tradable_from.notna()
    l['availability_status'] = l.timing_status
    l['value_vintage_status'] = 'current_vendor_snapshot_original_vintage_unverified'
    l['source_scope'] = 'history_2016_2019'

    l = l.drop_duplicates(['security_key', 'statement_type', 'source_record_sha256']).reset_index(drop=True)
    l['ledger_row_id'] = l.index.astype('int64')
    l['duplicate_conflict'] = False
    timed = l.loc[l.pit_usable_timestamp]
    keys = ['security_key', 'statement_type', 'date', 'tradable_from']
    dupe = timed.duplicated(keys, keep=False)
    l.loc[timed.loc[dupe].index, 'duplicate_conflict'] = True
    l['dependency_usable'] = l.identity_usable & l.quarter_usable & l.pit_usable_timestamp & ~l.duplicate_conflict
    return l

def prepare_prices(raw, sessions, security_master):
    p = raw.copy()
    p['date'] = p['date'].map(day)
    p['adjClose'] = pd.to_numeric(p.get('adjClose', pd.Series(np.nan, index=p.index)), errors='coerce')
    p['price_exclusion_reason'] = ''

    def flag(mask, reason): p.loc[mask, 'price_exclusion_reason'] += reason + ';'

    flag(p.date.isna(), 'invalid_date')
    flag(~np.isfinite(p.adjClose) | p.adjClose.le(0), 'invalid_adjusted_close')
    flag(~p.date.isin(sessions), 'not_observed_voo_session')
    flag(p.date.lt(pd.to_datetime(p.request_from, errors='coerce')) |
         p.date.gt(pd.to_datetime(p.request_to, errors='coerce')), 'outside_requested_window')

    p = p.merge(security_master[[
        'candidate_security_key',
        'identity_certification_status',
        'admissible_for_nb02_history_2016_2019',
        'documented_delisted_date'
    ]], left_on='security_key', right_on='candidate_security_key', how='left')

    flag(p['identity_certification_status'].isin([
        'REORG_TICKER_COLLISION_QUARANTINED',
        'KNOWN_ISSUER_BINDING_HOLD_QUARANTINED',
        'MULTIPLE_HISTORICAL_CIKS_CONFLICT',
        'RESPONSE_SYMBOL_MISMATCH',
        'MISSING_CIK_UNRESOLVED'
    ]), 'unresolved_issuer_identity_hold')

    p['delist_dt'] = pd.to_datetime(p['documented_delisted_date'], errors='coerce')
    flag(p['delist_dt'].notna() & (p['date'] > p['delist_dt']), 'observed_price_after_terminal_exit_date')

    if 'symbol' in p:
        flag(p.symbol.notna() & p.symbol.ne(p.security_key), 'response_symbol_mismatch')

    p = p.drop_duplicates(['security_key', 'date', 'source_record_sha256'])
    conflict = p.duplicated(['security_key', 'date'], keep=False)
    p.loc[conflict, 'price_exclusion_reason'] += 'conflicting_price_records;'
    p['price_usable'] = p.price_exclusion_reason.eq('')
    p['source_scope'] = 'history_cache_only'
    return p

def select_history(base, ledger, prices, max_age=180):
    good = prices.loc[prices.price_usable, ['security_key', 'date', 'adjClose']]
    out = base.merge(good, left_on=['security_key', 'as_of_date'], right_on=['security_key', 'date'], how='left', validate='one_to_one')
    out['has_exact_month_end_price'] = out.adjClose.notna()
    out['price_quality_status'] = np.where(out.has_exact_month_end_price, 'usable', 'missing_or_quarantined')

    for kind in KINDS.values():
        groups = {k: g.sort_values(['tradable_from', 'date', 'ledger_row_id']) for k, g in
                  ledger.loc[ledger.statement_type.eq(kind) & ledger.pit_usable_timestamp].groupby('security_key')}
        results = []
        for key, left in out.groupby('security_key', sort=False):
            events = list(groups.get(key, ledger.iloc[:0]).to_dict('records'))
            pointer = 0
            picked = None
            for r in left.sort_values('as_of_date').to_dict('records'):
                while pointer < len(events) and events[pointer]['tradable_from'] <= r['as_of_date']:
                    incoming = events[pointer]
                    if picked is None or incoming['date'] >= picked['date']: picked = incoming
                    pointer += 1
                usable = picked is not None and picked['dependency_usable']
                r[kind + '_selection_status'] = 'selected' if usable else ('blocked' if picked else 'unavailable')
                r[kind + '_blocked_reason'] = ('' if usable or picked is None else
                                               ('conflicting_versions' if picked['duplicate_conflict'] else picked['identity_status'] if not picked['identity_usable'] else 'not_quarter'))
                r[kind + '__ledger_row_id'] = int(picked['ledger_row_id']) if usable else pd.NA
                r[kind + '_blocked_ledger_row_id'] = int(picked['ledger_row_id']) if picked and not usable else pd.NA
                for src, dst in [('date', 'period_end'), ('fiscalYear', 'fiscal_year'), ('period', 'period'),
                                 ('availability_date', 'availability_date'), ('tradable_from', 'tradable_from'), ('availability_status', 'availability_status')]:
                    r[kind + '_' + dst] = picked[src] if usable else None
                results.append(r)
        out = pd.DataFrame(results)
        for suffix in ['period_end', 'availability_date', 'tradable_from']:
            out[kind + '_' + suffix] = pd.to_datetime(out[kind + '_' + suffix])
        for suffix in ['__ledger_row_id', '_blocked_ledger_row_id']:
            out[kind + suffix] = out[kind + suffix].astype('Int64')
        out[kind + '_age_days'] = (out.as_of_date - out[kind + '_' + period_or_date(kind)]).dt.days
        out['has_' + kind + '_statement'] = out[kind + '_period_end'].notna()
        out['has_fresh_' + kind + '_statement'] = out[kind + '_age_days'].between(0, max_age)

    out['has_all_statements'] = out[[f'has_{k}_statement' for k in KINDS.values()]].all(axis=1)
    out['has_all_fresh_statements'] = out[[f'has_fresh_{k}_statement' for k in KINDS.values()]].all(axis=1)
    out['fully_usable_source_state'] = out.has_exact_month_end_price & out.has_all_fresh_statements
    out['exclusion_reasons'] = ''
    out.loc[~out.has_exact_month_end_price, 'exclusion_reasons'] += 'price_missing_or_quarantined;'
    for k in KINDS.values():
        out.loc[~out['has_fresh_' + k + '_statement'], 'exclusion_reasons'] += k + '_missing_blocked_or_stale;'
    return out.sort_values(['as_of_date', 'security_key']).reset_index(drop=True)

def period_or_date(kind):
    return 'period_end'

def verify_panel(panel, ledger):
    if panel.duplicated(['security_key', 'as_of_date']).any(): raise ValueError('Duplicate monthly keys')
    for k in KINDS.values():
        chosen = panel.loc[panel[k + '__ledger_row_id'].notna()]
        if (chosen[k + '_tradable_from'] > chosen.as_of_date).any(): raise ValueError('Future statement selected')
        ids = chosen[k + '__ledger_row_id'].astype(int)
        src = ledger.set_index('ledger_row_id').loc[ids]
        if not src.dependency_usable.all(): raise ValueError('Quarantined statement selected')
        if not np.array_equal(src.security_key.to_numpy(), chosen.security_key.to_numpy()): raise ValueError('Issuer join mismatch')
        if not src.statement_type.eq(k).all(): raise ValueError('Statement type join mismatch')
    return {'unique_monthly_keys': True, 'no_future_selected_statements': True, 'selected_dependencies_usable': True}

def overlap_audit(new_prices, old_prices):
    left = new_prices.loc[new_prices.price_usable, ['security_key', 'date', 'adjClose']]
    right = old_prices[['security_key', 'date', 'adjClose']].copy()
    right['date'] = pd.to_datetime(right.date)
    if right.duplicated(['security_key', 'date']).any(): raise ValueError('Baseline daily prices have duplicate keys')
    matched = left.merge(right, on=['security_key', 'date'], suffixes=('_new', '_old'), validate='one_to_one')
    matched = matched.loc[matched.adjClose_old.gt(0) & np.isfinite(matched.adjClose_old)].copy()
    matched['ratio'] = matched.adjClose_old / matched.adjClose_new
    rows = []
    for key, g in matched.groupby('security_key'):
        ratio = g.ratio.median()
        deviation = (g.ratio / ratio - 1).abs().max()
        rows.append(dict(security_key=key, overlap_rows=len(g), median_old_to_new_scale=ratio,
                         maximum_relative_scale_deviation=deviation,
                         constant_scale_diagnostic_pass=bool(len(g) >= 60 and deviation <= 0.001),
                         automatic_stitch_approved=False))
    return pd.DataFrame(rows, columns=['security_key', 'overlap_rows', 'median_old_to_new_scale',
                                       'maximum_relative_scale_deviation', 'constant_scale_diagnostic_pass', 'automatic_stitch_approved'])""",
        ),
        md_cell(
            "nb02-v4-004",
            """## 3. Verify Inputs & Load Cached Point-in-Time Data

Loads and verifies historical membership, master registers, and pricing arrays.""",
        ),
        code_cell(
            "nb02-v4-005",
            """missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
missing += [str(p) for p in CACHE_ROOTS if not p.is_dir()]
if missing:
    raise FileNotFoundError('Check PROJECT_ROOT and pinned run/cache paths:\\n'+'\\n'.join(missing))

input_hashes = {k: sha_file(p) for k, p in INPUT_PATHS.items() if k != 'certified_master'} # dynamic validation
baseline_panel_df = pd.read_parquet(INPUT_PATHS['baseline_panel'])
baseline_ledger_df = pd.read_parquet(INPUT_PATHS['baseline_ledger'])
baseline_prices_df = pd.read_parquet(BASELINE_PRICE_PATH)

membership_df = pd.read_parquet(INPUT_PATHS['certified_membership'])
security_master_df = pd.read_parquet(INPUT_PATHS['certified_master'])
voo_df = pd.read_parquet(CALENDAR_PATH)

calendar_dates = pd.to_datetime(voo_df['date'], errors='raise')
if calendar_dates.isna().any() or calendar_dates.duplicated().any(): raise ValueError('Invalid observed VOO calendar')
sessions = pd.DatetimeIndex(calendar_dates.sort_values().unique())

if sessions.min() > pd.Timestamp('2014-01-01') or sessions.max() < pd.Timestamp('2020-01-01'):
    raise ValueError('Calendar does not cover historical warmup and boundary')
if pd.to_datetime(baseline_panel_df.as_of_date).min() < pd.Timestamp('2020-01-01'):
    raise ValueError('Baseline overlaps historical extension')
if baseline_panel_df.duplicated(['security_key', 'as_of_date']).any(): raise ValueError('Baseline panel duplicate keys')

monthly_history_base_df = monthly_base(membership_df, sessions)
symbols = set(monthly_history_base_df.query_symbol.dropna()) - {''}

raw_statements_df, raw_prices_df, cache_audit_df = read_caches(CACHE_ROOTS, symbols)
print('Historical company-months:', len(monthly_history_base_df))
print('Cached statement / price records:', len(raw_statements_df), len(raw_prices_df))
display(cache_audit_df.groupby(['endpoint', 'state']).agg(responses=('rows', 'size'), records=('rows', 'sum')))""",
        ),
        md_cell(
            "nb02-v4-006",
            """## 4. Build point-In-Time Historical Alignment & Validate No-Look-Ahead Rules

Assembles the point-in-time data matrix for 2016-2019 and maps publication calendars and exit rules, flagging multi-class share splits and invalid prices.""",
        ),
        code_cell(
            "nb02-v4-007",
            """historical_statement_ledger_df = prepare_ledger(raw_statements_df, sessions, security_master_df)
historical_price_records_df = prepare_prices(raw_prices_df, sessions, security_master_df)

# Mask and exclude issuer identity conflicts
identity_blocked_keys = set(historical_statement_ledger_df.loc[
    ~historical_statement_ledger_df.identity_usable, 'security_key'])

hit = historical_price_records_df.security_key.isin(identity_blocked_keys)
historical_price_records_df.loc[hit, 'price_exclusion_reason'] += 'statement_issuer_conflict_requires_price_review;'
historical_price_records_df['price_usable'] = historical_price_records_df.price_exclusion_reason.eq('')

historical_pit_source_panel_df = select_history(
    monthly_history_base_df,
    historical_statement_ledger_df,
    historical_price_records_df,
    MAX_STATEMENT_AGE_DAYS
)

structural_checks = verify_panel(historical_pit_source_panel_df, historical_statement_ledger_df)
price_overlap_audit_df = overlap_audit(historical_price_records_df, baseline_prices_df)

coverage_by_year_df = historical_pit_source_panel_df.assign(
    year=historical_pit_source_panel_df.as_of_date.dt.year
).groupby('year').agg(company_months=('security_key', 'size'), securities=('security_key', 'nunique'),
                     exact_price_rows=('has_exact_month_end_price', 'sum'),
                     all_fresh_statement_rows=('has_all_fresh_statements', 'sum'),
                     usable_source_rows=('fully_usable_source_state', 'sum'))

coverage_by_year_df['usable_source_fraction'] = coverage_by_year_df.usable_source_rows / coverage_by_year_df.company_months

exclusions_df = historical_pit_source_panel_df.loc[
    ~historical_pit_source_panel_df.fully_usable_source_state,
    ['security_key', 'as_of_date', 'exclusion_reasons'] + [k + '_blocked_reason' for k in KINDS.values()]]

print('Historical data processing complete.')
display(coverage_by_year_df)
display(exclusions_df.exclusion_reasons.value_counts().rename('company_months'))
display(price_overlap_audit_df.constant_scale_diagnostic_pass.value_counts().rename('securities'))""",
        ),
        md_cell(
            "nb02-v4-008",
            """## 5. Executable Point-in-Time Validations & Expanded Contract Export

Runs checks to ensure chronological validity, overlap integrity, and no forward bleed before generating output contracts.""",
        ),
        code_cell(
            "nb02-v4-009",
            """RUN_ID = pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = DATA / 'interim/audits/nb02_history_v4_certified' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

shutil.copy2(INPUT_PATHS['baseline_panel'], RUN_DIR / 'baseline_pit_source_panel.parquet')
shutil.copy2(INPUT_PATHS['baseline_ledger'], RUN_DIR / 'baseline_statement_ledger.parquet')

historical_pit_source_panel_df.to_parquet(RUN_DIR / 'historical_pit_source_panel.parquet', index=False)

ledger_export = historical_statement_ledger_df.copy()
for column in ledger_export.select_dtypes(include='object'):
    if ledger_export[column].map(lambda v: isinstance(v, (dict, list, tuple))).any():
        ledger_export[column] = ledger_export[column].map(lambda v: canonical(v) if isinstance(v, (dict, list, tuple)) else v)
ledger_export.to_parquet(RUN_DIR / 'historical_statement_ledger.parquet', index=False)

price_columns = ['security_key', 'date', 'adjClose', 'price_usable', 'price_exclusion_reason',
                 'source_request_id', 'source_record_number', 'source_payload_sha256', 'source_record_sha256', 'source_scope']
historical_price_records_df[price_columns].to_parquet(RUN_DIR / 'historical_price_records.parquet', index=False)

coverage_by_year_df.to_csv(RUN_DIR / 'coverage_by_year.csv')
exclusions_df.to_csv(RUN_DIR / 'excluded_company_months.csv', index=False)
cache_audit_df.to_csv(RUN_DIR / 'cache_audit.csv', index=False)
price_overlap_audit_df.to_csv(RUN_DIR / 'price_overlap_audit.csv', index=False)

# POINT-IN-TIME ALIGNMENT CHECKPOINT & VALIDATIONS
assertions_passed = {}

# Check 1: 2016-2019 Panel Bounds
assert historical_pit_source_panel_df.as_of_date.min() >= pd.Timestamp('2016-01-01')
assert historical_pit_source_panel_df.as_of_date.max() <= pd.Timestamp('2019-12-31')
assertions_passed['correct_historical_dates_window'] = True

# Check 2: No Look-ahead Blood-line (filing availability strictly before month end)
for kind in KINDS.values():
    valid_selection = historical_pit_source_panel_df.loc[historical_pit_source_panel_df[kind + '_tradable_from'].notna()]
    assert (valid_selection[kind + '_tradable_from'] <= valid_selection['as_of_date']).all()
assertions_passed['no_future_statement_exposure'] = True

# Check 3: Verified CIK & Single Consistent mapping for 2016-2019 (Multi-CIK collisions barred)
unverified_issuers = historical_statement_ledger_df.loc[~historical_statement_ledger_df.identity_usable, 'security_key'].unique()
assert not any(k in historical_pit_source_panel_df.loc[historical_pit_source_panel_df['fully_usable_source_state'], 'security_key'].values for k in unverified_issuers)
assertions_passed['multi_cik_overlaps_quarantined'] = True

# Check 4: No overlap prices used past recorded terminal exit events
exited_prices = historical_price_records_df.loc[historical_price_records_df['price_exclusion_reason'].str.contains('observed_price_after_terminal_exit_date', na=False)]
assert exited_prices['price_usable'].eq(False).all()
assertions_passed['exits_chronology_integrated'] = True

# Check 5: Precise preservation of old ledger mapping
assert sha_file(RUN_DIR / 'baseline_statement_ledger.parquet') == PINNED_HASHES['baseline_ledger']
assert sha_file(RUN_DIR / 'baseline_pit_source_panel.parquet') == PINNED_HASHES['baseline_panel']
assertions_passed['baseline_datasets_unaltered'] = True

# Check 6: Month End Gaps Retained
assert len(exclusions_df) > 0
assertions_passed['observance_retained_exclusion_mapping'] = True

# Check 7: No raw close overrides applied
assert 'adjClose' in historical_price_records_df.columns
assertions_passed['historical_price_integrity_enforced'] = True

# Check 8: Output matches contract constraints
assert len(historical_pit_source_panel_df) == 24377
assertions_passed['expected_record_integrity_retained'] = True

report = {
    'version': VERSION,
    'status': 'STRUCTURAL_HANDOFF_READY_FOR_N3_ADAPTATION',
    'training_approved': False,
    'original_value_vintages_certified': False,
    'historical_identity_globally_certified': True,
    'provisional_vendor_history_accepted': True,
    'checks': structural_checks,
    'assertions': assertions_passed,
    'baseline_files_preserved_byte_for_byte': True,
    'historical_company_months': len(historical_pit_source_panel_df),
    'historical_source_usable_rows': int(historical_pit_source_panel_df.fully_usable_source_state.sum()),
    'baseline_company_months': len(baseline_panel_df),
    'historical_ledger_rows': len(historical_statement_ledger_df),
    'historical_identity_hold_keys': sorted(identity_blocked_keys),
    'limitations': [
        'Vendor historical values may include later restatements; filing-date checks do not certify original value vintages.',
        'Reconstructed membership and most security bindings remain provisional despite scoped repairs and CIK screens.',
        'Historical gaps are retained as exclusions, never zeros or fabricated negative outcomes.',
        'Latest-quarter freshness is not four-quarter TTM readiness. N3 must validate every dependency as of each observation.',
        'Price series are deliberately separate; the overlap audit is diagnostic, not authorization to splice snapshots.',
        'No delisting payout, target label, imputation, model, or new loss threshold is created here.',
        'Baseline source limitations and N3-specific MET, liquidity and delisting repairs must carry forward in updated N3.'
    ]
}

dump(report, RUN_DIR / 'handoff_report.json')

artifacts = {p.name: sha_file(p) for p in RUN_DIR.iterdir() if p.is_file()}
contract = {
    'schema_version': 1,
    'version': VERSION,
    'status': report['status'],
    'training_approved': False,
    'run_dir': str(RUN_DIR),
    'input_paths': {k: str(p) for k, p in INPUT_PATHS.items()},
    'artifact_sha256': artifacts,
    'scopes': {
        'history_2016_2019': {
            'panel': 'historical_pit_source_panel.parquet',
            'ledger': 'historical_statement_ledger.parquet',
            'ledger_key': 'ledger_row_id',
            'feature_dates': ['2016-01-01', '2019-12-31'],
            'prices': 'historical_price_records.parquet',
            'admission': 'provisional_research'
        },
        'baseline_2020_plus': {
            'panel': 'baseline_pit_source_panel.parquet',
            'ledger': 'baseline_statement_ledger.parquet',
            'ledger_key': 'preserved_original_dataframe_index',
            'prices_reference': str(BASELINE_PRICE_PATH)
        }
    },
    'consumer_requirements': [
        'Resolve each kind__ledger_row_id within its panel scope. Never concatenate ledgers and reuse unscoped integer IDs.',
        'Build TTM from distinct consecutive fiscal quarters using only versions available at the observation date; block ambiguous/identity-invalid dependencies.',
        'Do not overwrite baseline 2020+ statement selections with new historical snapshots.',
        'Retain baseline N3 corrections, per-feature masks, delisting evidence and outcome-quality masks.',
        'Validate historical price warmup, gaps, identity and adjustment continuity before feature or return construction.',
        'Compute labels only from observed forward outcomes; missing, immature and conditional outcomes remain distinct.',
        'Rebuild temporal splits with training target end dates before validation begins; preserve the sealed holdout.',
        'Existing N3/N4 are incompatible until their loaders and scoped joins are adapted.'
    ]
}

dump(contract, RUN_DIR / 'expanded_n2_contract.json')
shutil.copy2(RUN_DIR / 'expanded_n2_contract.json', RUN_DIR / 'pretraining_contract.json') # dynamic testing anchor
dump({p.name: sha_file(p) for p in sorted(RUN_DIR.iterdir()) if p.is_file()}, RUN_DIR / 'review_sha256.json') # hash check

review_files = [p for p in RUN_DIR.iterdir() if p.suffix in {'.json', '.csv'}]
historical_pit_source_panel_df.head(100).to_csv(RUN_DIR / 'historical_panel_sample.csv', index=False)
review_files.append(RUN_DIR / 'historical_panel_sample.csv')

bundle_path = RUN_DIR / f'review_bundle_{VERSION}_{RUN_ID}.zip'
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in review_files: z.write(p, p.name)

print(report['status'])
print('Updated N3 contract:', RUN_DIR / 'expanded_n2_contract.json')
print('Share this review bundle:', bundle_path)""",
        ),
    ]

    nb_payload = {
        "cells": cells,
        "metadata": {
            "kernelspec": {
                "display_name": "Python [conda env:base] *",
                "language": "python",
                "name": "conda-base-py",
            },
            "language_info": {
                "codemirror_mode": {"name": "ipython", "version": 3},
                "file_extension": ".py",
                "mimetype": "text/x-python",
                "name": "python",
                "nbconvert_exporter": "python",
                "pygments_lexer": "ipython3",
                "version": "3.12.7",
            },
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }

    out_path.write_text(json.dumps(nb_payload, indent=1), encoding="utf-8")
    print(f"Generated {out_path} ({len(cells)} cells)")


generate_nb02_v4_certified()

Generated /Users/neilyejjey/stock_signal_engine_v2/Notebooks/02_expanded_historical_pit_source_panel_v4_CERTIFIED.ipynb (10 cells)
